# DentaMind AI - CV5 candidate path PRE-FLIGHT (no training, no DENTEX data)

Exercises the candidate notebook path of `kaggle/make_cv5_notebook.py` on Kaggle: isolated training subprocesses, section-9 retry
rules, per-run training manifests, run-status serialization, the run-evidence archive, and the gate-4 stop of aggregation.

* **No input dataset may be attached.** No DENTEX image, label, split manifest or fold file is read; the frame is synthetic.
* Training is replaced by `cv5_preflight_trainer.py`, which saves **untrained, randomly initialised** ResNet-50 weights. No model is
  trained and no number produced here is a result. Every run is recorded as pre-flight and is never eligible for final-model selection.
* Scripted outcomes: {"CV5_PREFLIGHT_f1_s43": "infra_once", "CV5_PREFLIGHT_f2_s44": "compute", "CV5_PREFLIGHT_f2_s43": "inf_loss", "CV5_PREFLIGHT_f3_s45": "collapse", "CV5_PREFLIGHT_f3_s42": "ambiguous_cuda", "CV5_PREFLIGHT_f4_s44": "kill_after_checkpoint", "CV5_PREFLIGHT_f4_s43": "killed"}; seed 46 is never started.
* Output: `dentamind_cv5_preflight_output.zip` with `preflight_report.json` (PASS/FAIL per check).

In [ ]:
# ---- configuration -----------------------------------------------------------------------------------------------
import os, sys, json, time, shutil, subprocess, glob, zipfile, hashlib, dataclasses, collections
from pathlib import Path
import numpy as np

SEEDS = [42, 43, 44, 45, 46]          # protocol section 3
FOLDS = [0, 1, 2, 3, 4]               # all five folds; never reduced
TIME_BUDGET_H = 11.0                  # Kaggle sessions stop at 12 h; no new run starts after this budget
RESUME_FROM = None                    # e.g. "/kaggle/input/<this notebook's previous output>/cv5/runs"

# Local pipeline check only (never on Kaggle): DENTAMIND_SMOKE=1 -> 1 epoch, seed 42, all 5 folds; output marked SMOKE.
SMOKE = os.environ.get("DENTAMIND_SMOKE") == "1"
if SMOKE:
    SEEDS = [42]

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path(os.environ.get("DENTAMIND_WORK") or ("/kaggle/working" if ON_KAGGLE else "kaggle_work_cv5")).resolve()
ART = WORK / "cv5"
RUNS = ART / "runs"
DATA_ROOT = Path(os.environ.get("DENTAMIND_DATA_ROOT") or ("/tmp/dentamind_data" if ON_KAGGLE else WORK / "data"))
CODE = WORK / "code"
CACHE = Path("/tmp/dentamind_cache_cv5") if ON_KAGGLE else WORK / "cache"
for d in (ART, RUNS, DATA_ROOT, CODE, CACHE):
    d.mkdir(parents=True, exist_ok=True)
T0 = time.time()
LOG = ART / "training_log.txt"

def log(*a):
    s = " ".join(str(x) for x in a)
    print(s, flush=True)
    with open(LOG, "a", encoding="utf-8") as fh:
        fh.write(time.strftime("%Y-%m-%d %H:%M:%S ") + s + "\n")

if RESUME_FROM and Path(RESUME_FROM).is_dir():
    shutil.copytree(RESUME_FROM, RUNS, dirs_exist_ok=True)
    log("resumed run folders from", RESUME_FROM)
log("CV5", "SMOKE (pipeline check, not a result)" if SMOKE else "full run", "seeds", SEEDS, "folds", FOLDS)

In [ ]:
# ---- 1. GPU and software ----------------------------------------------------------------------------------------------
import numpy as np, PIL, platform, torch, torchvision
env = {"python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__, "numpy": np.__version__,
       "pillow": PIL.__version__, "cuda_available": torch.cuda.is_available(), "cuda": torch.version.cuda,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "gpu_count": torch.cuda.device_count(),
       "disk_free_gb_working": round(shutil.disk_usage(WORK).free / 1e9, 1), "disk_free_gb_data": round(shutil.disk_usage(DATA_ROOT).free / 1e9, 1),
       "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
try:
    env["nvidia_smi"] = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                                       capture_output=True, text=True, timeout=20).stdout.strip()
except Exception as e:
    env["nvidia_smi"] = f"unavailable: {e}"
log(json.dumps(env, indent=1))
(ART / "environment.json").write_text(json.dumps(env, indent=1))
if not torch.cuda.is_available():
    log("WARNING: no GPU. Enable Accelerator = GPU; 35+ ResNet-50 runs on CPU are not practical.")

In [ ]:
# ---- P1. embedded code (no bundle, no data), candidate definition, pre-flight trainer ---------------------------------------
inputs = sorted(glob.glob("/kaggle/input/*"))
log("attached inputs:", inputs or "none")
assert not inputs, "PRE-FLIGHT: detach every input dataset - no DENTEX data may be available to this notebook"
EMBEDDED = {"kaggle/dentamind_kaggle.py": "\"\"\"DentaMind finding-classifier imbalance experiments: data, training (resumable), evaluation, acceptance rule.\r\n\r\nUsed by kaggle/denta_mind_training.ipynb; importable locally too (tests/test_kaggle_training_lib.py). Depends on torch, torchvision,\r\nnumpy, Pillow and scripts/dataset/dsutil.py (crop rule identical to production ``trainers.crop_region``).\r\n\r\nThe baseline reproduces the project's same-code baseline C0r (= scripts/ml_cls_train.py --exp C0r): ResNet-50 IMAGENET1K_V2,\r\n224 px ground-truth crops (pad 0.10), AdamW lr 3e-4 wd 1e-4, cosine T_max 15, batch 32, 15 epochs, horizontal flip p=0.5 +\r\n+-5 % contrast, inverse-frequency class weights capped at 5, best epoch by validation macro-F1. Exactly one factor changes per\r\nexperiment. The test split is evaluated only for the baseline and for candidates that pass the acceptance rule on validation.\r\n\"\"\"\r\nfrom __future__ import annotations\r\n\r\nimport copy\r\nimport hashlib\r\nimport json\r\nimport math\r\nimport os\r\nimport shutil\r\nimport sys\r\nimport time\r\nfrom dataclasses import asdict, dataclass, field\r\nfrom pathlib import Path\r\n\r\nimport numpy as np\r\n\r\nHERE = Path(__file__).resolve().parent\r\nfor p in (HERE.parent / \"scripts\" / \"dataset\", HERE / \"scripts\" / \"dataset\"):\r\n    if p.is_dir() and str(p) not in sys.path:\r\n        sys.path.insert(0, str(p))\r\nfrom dsutil import CLASSES, CROP_PAD, CROP_SIZE, crop_region, read_csv, sha256_file  # noqa: E402\r\n\r\nMEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)\r\nIOU_THR, DET_SCORE = 0.5, 0.3                      # same as scripts/ml_recipe_eval.py\r\nABSTAIN_ENTROPY = 0.3965956833419285               # production rule (data/reports/uncertainty_thresholds.json), NOT refitted\r\n# Pre-registered acceptance criteria of this project (data/reports/training_recipe_comparison_protocol.md section 5, and the\r\n# precision side-condition of docs/audit_evidence/ml_improvement/imbalance/report.md). Applied unchanged; candidate vs BASE.\r\nACCEPT = {\"gain_min\": 0.027, \"seed_pairs_min\": 4, \"cross_pairs_min\": 20, \"class_f1_drop_max\": 0.05, \"gt_macro_f1_drop_max\": 0.02,\r\n          \"minority_precision_drop_max\": 0.10}\r\nMINORITY = (\"Deep Caries\", \"Periapical Lesion\")\r\n\r\n\r\n# ----------------------------------------------------------------------------------------------------------------- config\r\n@dataclass\r\nclass ExpConfig:\r\n    name: str\r\n    loss: str = \"ce\"                    # ce | focal\r\n    weights: str = \"inverse_cap5\"       # none | inverse_cap5 | sqrt | effective_number\r\n    sampler: str = \"none\"               # none | sqrt_inverse (p ~ n_c^-0.5, epoch length unchanged)\r\n    data: str = \"dentex\"                # dentex | dentex+official_val\r\n    gamma: float = 2.0\r\n    beta: float = 0.999                 # effective-number weights\r\n    epochs: int = 15\r\n    batch_size: int = 32\r\n    lr: float = 3e-4\r\n    wd: float = 1e-4\r\n    hflip: float = 0.5\r\n    contrast: float = 0.1               # multiplicative +-contrast/2 around the per-image mean, clamped to [0,1]\r\n    note: str = \"\"\r\n    extra: dict = field(default_factory=dict)\r\n\r\n\r\nEXPERIMENTS = {\r\n    \"REF_unweighted\": ExpConfig(\"REF_unweighted\", weights=\"none\", note=\"reference: plain cross-entropy\"),\r\n    \"BASE\": ExpConfig(\"BASE\", note=\"current recipe (C0r): inverse-frequency weights capped at 5\"),\r\n    \"A1_sqrt_weights\": ExpConfig(\"A1_sqrt_weights\", weights=\"sqrt\", note=\"Strategy A: sqrt(inverse frequency) weights\"),\r\n    \"A2_effnum_weights\": ExpConfig(\"A2_effnum_weights\", weights=\"effective_number\", note=\"Strategy A: effective-number weights (Cui 2019, beta 0.999)\"),\r\n    \"B_sampler\": ExpConfig(\"B_sampler\", weights=\"none\", sampler=\"sqrt_inverse\", note=\"Strategy B: weighted sampler p~n_c^-0.5, plain CE\"),\r\n    \"C_focal\": ExpConfig(\"C_focal\", loss=\"focal\", weights=\"none\", note=\"Strategy C: focal loss gamma 2, no class weights\"),\r\n    \"D_official_val\": ExpConfig(\"D_official_val\", data=\"dentex+official_val\", note=\"Strategy D: BASE + DENTEX official-validation findings in train\"),\r\n    # external panoramic data (reports/external_data_protocol.md, pre-registered 2026-10-24); BASE recipe, only the data changes\r\n    \"X1_cluj\": ExpConfig(\"X1_cluj\", data=\"dentex+cluj\", note=\"BASE + Cluj (Zenodo 15487430) Impacted, Periapical Lesion, partial caries crops\"),\r\n    \"X2_hanoi\": ExpConfig(\"X2_hanoi\", data=\"dentex+hanoi\", note=\"BASE + Hanoi (Mendeley kx52tk2ddj) Periapical Lesion crops, cap 1,500\"),\r\n    \"X3_both\": ExpConfig(\"X3_both\", data=\"dentex+cluj+hanoi\", note=\"BASE + Cluj + Hanoi (cap 1,500)\"),\r\n}\r\nPARTIAL_CARIES = 4          # label index: {Caries, Deep Caries} (caries of unrecorded depth); never a prediction target\r\n\r\n\r\ndef class_counts(y):\r\n    \"\"\"Counts of the four fully labelled classes (partial labels excluded, so BASE weights are unchanged).\"\"\"\r\n    y = np.asarray(y)\r\n    return np.bincount(y[y < len(CLASSES)], minlength=len(CLASSES))\r\n\r\n\r\ndef ce_partial(torch, logits, target, weight=None):\r\n    \"\"\"Weighted mean cross-entropy that also accepts the partial label PARTIAL_CARIES: loss = -log(p_Caries + p_DeepCaries).\r\n    With no partial label in the batch it is exactly F.cross_entropy(logits, target, weight=weight). Partial samples get the\r\n    Caries class weight (protocol section 1).\"\"\"\r\n    F = torch.nn.functional\r\n    part = target == PARTIAL_CARIES\r\n    if not bool(part.any()):\r\n        return F.cross_entropy(logits, target, weight=weight)\r\n    logp = F.log_softmax(logits, 1)\r\n    full = ~part\r\n    tgt = torch.where(full, target, torch.zeros_like(target))\r\n    nll = -logp.gather(1, tgt.view(-1, 1)).squeeze(1)\r\n    nll = torch.where(part, -torch.logsumexp(logp[:, :2], 1), nll)\r\n    w = weight[tgt] if weight is not None else torch.ones_like(nll)\r\n    return (w * nll).sum() / w.sum()\r\n\r\n\r\n# ------------------------------------------------------------------------------------------------------------------ data\r\ndef load_split(split_dir, split):\r\n    p = Path(split_dir) / f\"combined_{split}.csv\"\r\n    return read_csv(p) if p.is_file() else []\r\n\r\n\r\ndef verify_images(rows, data_root, max_report=10):\r\n    \"\"\"SHA-256 of every distinct image against the manifest. Returns (n_ok, problems).\"\"\"\r\n    seen, bad = {}, []\r\n    for r in rows:\r\n        if r[\"image_path\"] in seen:\r\n            continue\r\n        p = Path(data_root) / r[\"image_path\"]\r\n        if not p.is_file():\r\n            bad.append(f\"missing {r['image_path']}\")\r\n            seen[r[\"image_path\"]] = False\r\n            continue\r\n        ok = sha256_file(p) == r[\"image_sha256\"]\r\n        seen[r[\"image_path\"]] = ok\r\n        if not ok:\r\n            bad.append(f\"sha256 mismatch {r['image_path']}\")\r\n    return sum(seen.values()), bad[:max_report] + ([f\"... {len(bad) - max_report} more\"] if len(bad) > max_report else [])\r\n\r\n\r\ndef build_crops(rows, data_root, cache_file=None):\r\n    \"\"\"GT-box crops exactly as the production trainer makes them. -> X uint8 (N,224,224,3), y (N,), meta list.\"\"\"\r\n    from PIL import Image\r\n    if cache_file and Path(cache_file).is_file():\r\n        z = np.load(cache_file, allow_pickle=False)\r\n        meta = json.loads(Path(str(cache_file) + \".json\").read_text(encoding=\"utf-8\"))\r\n        if [m[\"sample_id\"] for m in meta] == [r[\"sample_id\"] for r in rows]:\r\n            return z[\"X\"], z[\"y\"], meta\r\n    by_img = {}\r\n    for i, r in enumerate(rows):\r\n        by_img.setdefault(r[\"image_path\"], []).append(i)\r\n    X = np.zeros((len(rows), CROP_SIZE, CROP_SIZE, 3), np.uint8)\r\n    keep = np.zeros(len(rows), bool)\r\n    for path, idx in by_img.items():\r\n        img = np.asarray(Image.open(Path(data_root) / path).convert(\"RGB\"))\r\n        for i in idx:\r\n            x, y_, w, h = json.loads(rows[i][\"bbox_xywh\"])\r\n            c = crop_region(img, (x, y_, x + w, y_ + h), CROP_SIZE, CROP_PAD)\r\n            if c is not None:\r\n                X[i], keep[i] = c, True\r\n    if not keep.all():\r\n        raise ValueError(f\"{(~keep).sum()} manifest rows produced no crop - manifest and crop rule disagree\")\r\n    y = np.array([int(r[\"class_index\"]) for r in rows], np.int64)\r\n    meta = [{\"sample_id\": r[\"sample_id\"], \"source_dataset\": r[\"source_dataset\"], \"image_path\": r[\"image_path\"],\r\n             \"radiograph_group\": r[\"radiograph_group\"]} for r in rows]\r\n    if cache_file:\r\n        np.savez(cache_file, X=X, y=y)\r\n        Path(str(cache_file) + \".json\").write_text(json.dumps(meta), encoding=\"utf-8\")\r\n    return X, y, meta\r\n\r\n\r\ndef box_iou(a, b):\r\n    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])\r\n    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)\r\n    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter\r\n    return float(inter / union) if union > 0 else 0.0\r\n\r\n\r\ndef match(gt, pred, thr=IOU_THR):\r\n    \"\"\"Greedy one-to-one matching by IoU (identical to scripts/ml_recipe_eval.py).\"\"\"\r\n    pairs = sorted(((box_iou(g, p), i, j) for i, g in enumerate(gt) for j, p in enumerate(pred)), reverse=True)\r\n    ug, up, m = set(), set(), {}\r\n    for v, i, j in pairs:\r\n        if v < thr:\r\n            break\r\n        if i in ug or j in up:\r\n            continue\r\n        ug.add(i)\r\n        up.add(j)\r\n        m[i] = (j, v)\r\n    return m\r\n\r\n\r\ndef build_detector_crops(rows, data_root, det_cache_path, image_id_of):\r\n    \"\"\"Crops from cached production-detector boxes (score >= 0.3) matched to GT findings. -> Xd, matched index array, n_missed.\r\n    ``image_id_of(row)`` maps a manifest row to the key used in the detector cache (project image_id).\"\"\"\r\n    from PIL import Image\r\n    det = json.loads(Path(det_cache_path).read_text(encoding=\"utf-8\"))\r\n    by_img = {}\r\n    for i, r in enumerate(rows):\r\n        by_img.setdefault(r[\"image_path\"], []).append(i)\r\n    Xd, mi, missed = [], [], 0\r\n    for path, idx in by_img.items():\r\n        key = image_id_of(rows[idx[0]])\r\n        boxes = [d[:4] for d in det.get(key, []) if d[4] >= DET_SCORE]\r\n        gts = []\r\n        for i in idx:\r\n            x, y_, w, h = json.loads(rows[i][\"bbox_xywh\"])\r\n            gts.append([x, y_, x + w, y_ + h])\r\n        m = match(gts, boxes)\r\n        img = None\r\n        for k, i in enumerate(idx):\r\n            hit = m.get(k)\r\n            if hit is None:\r\n                missed += 1\r\n                continue\r\n            img = img if img is not None else np.asarray(Image.open(Path(data_root) / path).convert(\"RGB\"))\r\n            c = crop_region(img, boxes[hit[0]], CROP_SIZE, CROP_PAD)\r\n            if c is None:\r\n                missed += 1\r\n                continue\r\n            Xd.append(c)\r\n            mi.append(i)\r\n    order = np.argsort(mi, kind=\"stable\")\r\n    return np.stack(Xd)[order], np.asarray(mi)[order], missed\r\n\r\n\r\n# ------------------------------------------------------------------------------------------------- imbalance components\r\ndef class_weights(counts, scheme, cap=5.0, beta=0.999):\r\n    \"\"\"Training-split class weights normalised so the average SAMPLE weight is 1. scheme: none | inverse_cap5 | sqrt | effective_number.\r\n    ``inverse_cap5`` equals scripts/ml_cls_train.class_weights(counts, 'weights').\"\"\"\r\n    counts = np.asarray(counts, float)\r\n    if (counts <= 0).any():\r\n        raise ValueError(f\"every class needs training samples, got {counts.tolist()}\")\r\n    if scheme == \"none\":\r\n        return None\r\n    if scheme in (\"inverse_cap5\", \"sqrt\"):\r\n        w = counts.sum() / (len(counts) * counts)               # inverse frequency (already average sample weight 1)\r\n        if scheme == \"sqrt\":\r\n            w = np.sqrt(w)\r\n    elif scheme == \"effective_number\":\r\n        w = (1 - beta) / (1 - np.power(beta, counts))\r\n        w = w / (w * counts).sum() * counts.sum()               # put on the same scale before the cap\r\n    else:\r\n        raise ValueError(scheme)\r\n    w = np.minimum(w, cap)\r\n    return w / (w * counts).sum() * counts.sum()\r\n\r\n\r\ndef sampler_indices(y, rng, power=0.5):\r\n    \"\"\"One epoch of indices drawn with replacement, p(sample) ~ n_class^-power; epoch length = len(y).\"\"\"\r\n    n = np.bincount(y, minlength=len(CLASSES)).astype(float)\r\n    p = (1.0 / n ** power)[y]\r\n    return rng.choice(len(y), size=len(y), replace=True, p=p / p.sum())\r\n\r\n\r\ndef sampler_exposure(y, power=0.5):\r\n    \"\"\"Expected draws per epoch of one sample of each class (1.0 = as often as without a sampler).\"\"\"\r\n    n = np.bincount(y, minlength=len(CLASSES)).astype(float)\r\n    p_class = n ** (1 - power) / (n ** (1 - power)).sum()\r\n    return {c: round(float(len(y) * p_class[k] / n[k]), 3) for k, c in enumerate(CLASSES)}\r\n\r\n\r\ndef focal_loss(torch, logits, target, gamma=2.0, weight=None):\r\n    ce = torch.nn.functional.cross_entropy(logits, target, reduction=\"none\", weight=weight)\r\n    pt = torch.exp(-torch.nn.functional.cross_entropy(logits, target, reduction=\"none\"))\r\n    return ((1 - pt) ** gamma * ce).mean()\r\n\r\n\r\ndef augment(torch, xb_u8, gen, cfg: ExpConfig, device):\r\n    \"\"\"Baseline augmentation of scripts/ml_cls_train.py ('baseline'): hflip p, multiplicative contrast 1+c*(u-0.5) around the\r\n    per-image mean, clamp [0,1]. Crop-level only (finding class is invariant to a mirror of the crop; no FDI task involved).\"\"\"\r\n    B = xb_u8.shape[0]\r\n    x = xb_u8.permute(0, 3, 1, 2).contiguous().float() / 255.0\r\n    r = lambda *s: torch.rand(*s, generator=gen)  # noqa: E731\r\n    if cfg.hflip:\r\n        f = (r(B) < cfg.hflip)\r\n        x = torch.where(f.view(-1, 1, 1, 1).to(device), x.flip(3), x)\r\n    if cfg.contrast:\r\n        cm = 1 + cfg.contrast * (r(B) - 0.5)\r\n        m = x.mean(dim=(1, 2, 3), keepdim=True)\r\n        r(B)    # brightness draw (amplitude 0 in the baseline); kept so the random stream matches scripts/ml_cls_train.py\r\n        x = ((x - m) * cm.view(-1, 1, 1, 1).to(device) + m).clamp(0, 1)\r\n    return x\r\n\r\n\r\ndef norm(torch, x01, device):\r\n    return (x01 - torch.tensor(MEAN, device=device).view(1, 3, 1, 1)) / torch.tensor(STD, device=device).view(1, 3, 1, 1)\r\n\r\n\r\n# ------------------------------------------------------------------------------------------------------------- metrics\r\ndef report(y_true, y_pred, prob=None):\r\n    n = len(CLASSES)\r\n    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)\r\n    cm = np.zeros((n, n), int)\r\n    for t, p in zip(y_true, y_pred):\r\n        cm[t, p] += 1\r\n    sup, tp = cm.sum(1), np.diag(cm)\r\n    prec = np.divide(tp, cm.sum(0), out=np.zeros(n), where=cm.sum(0) > 0)\r\n    rec = np.divide(tp, sup, out=np.zeros(n), where=sup > 0)\r\n    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(n), where=(prec + rec) > 0)\r\n    out = {\"n\": int(len(y_true)), \"accuracy\": float(tp.sum() / max(1, cm.sum())), \"macro_precision\": float(prec.mean()),\r\n           \"macro_recall\": float(rec.mean()), \"macro_f1\": float(f1.mean()), \"balanced_accuracy\": float(rec.mean()),\r\n           \"per_class\": {CLASSES[i]: {\"precision\": float(prec[i]), \"recall\": float(rec[i]), \"f1\": float(f1[i]), \"support\": int(sup[i]),\r\n                                      \"false_positives\": int(cm[:, i].sum() - cm[i, i])} for i in range(n)},\r\n           \"confusion\": cm.tolist()}\r\n    if prob is not None:\r\n        prob = np.asarray(prob, float)\r\n        conf, pred = prob.max(1), prob.argmax(1)\r\n        ece = 0.0\r\n        for b in range(10):\r\n            m = (conf > b / 10) & (conf <= (b + 1) / 10)\r\n            if m.any():\r\n                ece += m.mean() * abs((pred[m] == y_true[m]).mean() - conf[m].mean())\r\n        out[\"nll\"] = float(-np.log(np.clip(prob[np.arange(len(y_true)), y_true], 1e-9, 1)).mean())\r\n        out[\"ece_10bin\"] = float(ece)\r\n        H = -(np.clip(prob, 1e-12, 1) * np.log(np.clip(prob, 1e-12, 1))).sum(1) / math.log(n)\r\n        acc = H < ABSTAIN_ENTROPY\r\n        out[\"abstention_production_rule\"] = {\r\n            \"rule\": f\"abstain if normalised entropy >= {ABSTAIN_ENTROPY:.4f} (fitted for the deployed model; not refitted)\",\r\n            \"accepted\": int(acc.sum()), \"abstained\": int((~acc).sum()),\r\n            \"accuracy_on_accepted\": float((pred[acc] == y_true[acc]).mean()) if acc.any() else None}\r\n    return out\r\n\r\n\r\n# ------------------------------------------------------------------------------------------------------------- training\r\ndef software():\r\n    import platform\r\n    import torch\r\n    import torchvision\r\n    return {\"python\": platform.python_version(), \"torch\": torch.__version__, \"torchvision\": torchvision.__version__,\r\n            \"numpy\": np.__version__, \"cuda\": torch.version.cuda, \"gpu\": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,\r\n            \"platform\": platform.platform()}\r\n\r\n\r\ndef new_model(torch, tv, pretrained=True):\r\n    m = tv.models.resnet50(weights=tv.models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)\r\n    m.fc = torch.nn.Linear(m.fc.in_features, len(CLASSES))\r\n    return m\r\n\r\n\r\ndef predict(torch, model, X, device, bs=64):\r\n    model.eval()\r\n    out = []\r\n    with torch.no_grad():\r\n        for i in range(0, len(X), bs):\r\n            xb = torch.from_numpy(np.ascontiguousarray(X[i:i + bs])).to(device).permute(0, 3, 1, 2).float() / 255.0\r\n            out.append(torch.softmax(model(norm(torch, xb, device)).float(), 1).cpu().numpy())   # fp32 inference\r\n    return np.concatenate(out)\r\n\r\n\r\ndef free_disk_gb(path=\"/kaggle/working\"):\r\n    p = path if os.path.exists(path) else \".\"\r\n    return round(shutil.disk_usage(p).free / 1e9, 1)\r\n\r\n\r\ndef gpu_status(torch):\r\n    if not torch.cuda.is_available():\r\n        return {\"gpu\": None}\r\n    free, total = torch.cuda.mem_get_info()\r\n    return {\"gpu\": torch.cuda.get_device_name(0), \"mem_free_mb\": free // 2**20, \"mem_total_mb\": total // 2**20,\r\n            \"peak_alloc_mb\": torch.cuda.max_memory_allocated() // 2**20}\r\n\r\n\r\ndef train_run(cfg: ExpConfig, seed: int, train, val, out_dir, log=print):\r\n    \"\"\"Train one (experiment, seed). Resumable: per-epoch state in <out>/state.pt; a finished run (result.json) is skipped.\r\n    train/val: dicts with X, y (val also meta). Returns the result dict (validation only).\"\"\"\r\n    import torch\r\n    import torchvision as tv\r\n    out = Path(out_dir) / f\"{cfg.name}_s{seed}\"\r\n    out.mkdir(parents=True, exist_ok=True)\r\n    if (out / \"result.json\").is_file():\r\n        log(f\"[{cfg.name} s{seed}] already finished - skipped\")\r\n        return json.loads((out / \"result.json\").read_text(encoding=\"utf-8\"))\r\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\r\n    torch.backends.cudnn.benchmark = False\r\n    torch.backends.cudnn.deterministic = True\r\n    Xtr, ytr, Xva, yva = train[\"X\"], train[\"y\"], val[\"X\"], val[\"y\"]\r\n    counts = class_counts(ytr)\r\n    if (ytr == PARTIAL_CARIES).any() and (cfg.loss != \"ce\" or cfg.sampler != \"none\"):\r\n        raise ValueError(\"partial caries labels are only supported with plain (weighted) cross-entropy and no sampler\")\r\n    w = class_weights(counts, cfg.weights, beta=cfg.beta)\r\n    cw = torch.tensor(w, dtype=torch.float32, device=device) if w is not None else None\r\n    torch.manual_seed(seed)\r\n    np.random.seed(seed)\r\n    rng = np.random.default_rng(seed)\r\n    gen = torch.Generator().manual_seed(seed)\r\n    model = new_model(torch, tv).to(device)\r\n    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)\r\n    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, cfg.epochs))\r\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=device.type == \"cuda\")\r\n    start, best, best_ep, best_state, hist = 1, -1.0, 0, None, []\r\n    st = out / \"state.pt\"\r\n    if st.is_file():\r\n        s = torch.load(st, map_location=device, weights_only=False)\r\n        model.load_state_dict(s[\"model\"])\r\n        opt.load_state_dict(s[\"opt\"])\r\n        sched.load_state_dict(s[\"sched\"])\r\n        scaler.load_state_dict(s[\"scaler\"])\r\n        rng.bit_generator.state = s[\"rng\"]\r\n        gen.set_state(s[\"gen\"])\r\n        torch.set_rng_state(s[\"torch_rng\"])\r\n        start, best, best_ep, best_state, hist = s[\"epoch\"] + 1, s[\"best\"], s[\"best_ep\"], s[\"best_state\"], s[\"hist\"]\r\n        log(f\"[{cfg.name} s{seed}] resumed after epoch {s['epoch']}\")\r\n    if device.type == \"cuda\":\r\n        torch.cuda.reset_peak_memory_stats()\r\n    t0 = time.time()\r\n    for ep in range(start, cfg.epochs + 1):\r\n        model.train()\r\n        idx = sampler_indices(ytr, rng) if cfg.sampler == \"sqrt_inverse\" else rng.permutation(len(ytr))\r\n        tot, seen = 0.0, 0\r\n        for i in range(0, len(idx), cfg.batch_size):\r\n            b = idx[i:i + cfg.batch_size]\r\n            if len(b) < 2:\r\n                continue\r\n            xb = augment(torch, torch.from_numpy(Xtr[b]).to(device), gen, cfg, device)\r\n            yb = torch.from_numpy(ytr[b]).to(device)\r\n            opt.zero_grad(set_to_none=True)\r\n            with torch.autocast(device_type=device.type, enabled=device.type == \"cuda\"):\r\n                lo = model(norm(torch, xb, device))\r\n            lo = lo.float()\r\n            loss = focal_loss(torch, lo, yb, cfg.gamma, cw) if cfg.loss == \"focal\" else ce_partial(torch, lo, yb, cw)\r\n            scaler.scale(loss).backward()\r\n            scaler.step(opt)\r\n            scaler.update()\r\n            tot += float(loss.detach()) * len(b)\r\n            seen += len(b)\r\n        sched.step()\r\n        prob = predict(torch, model, Xva, device)\r\n        rep = report(yva, prob.argmax(1), prob)\r\n        h = {\"epoch\": ep, \"train_loss\": tot / max(1, seen), \"val_nll\": rep[\"nll\"], \"val_macro_f1\": rep[\"macro_f1\"],\r\n             \"val_acc\": rep[\"accuracy\"], \"val_dc_recall\": rep[\"per_class\"][\"Deep Caries\"][\"recall\"],\r\n             \"val_pl_recall\": rep[\"per_class\"][\"Periapical Lesion\"][\"recall\"], \"sec\": round(time.time() - t0, 1)}\r\n        hist.append(h)\r\n        if rep[\"macro_f1\"] > best:\r\n            best, best_ep, best_state = rep[\"macro_f1\"], ep, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\r\n        torch.save({\"model\": model.state_dict(), \"opt\": opt.state_dict(), \"sched\": sched.state_dict(), \"scaler\": scaler.state_dict(),\r\n                    \"rng\": rng.bit_generator.state, \"gen\": gen.get_state(), \"torch_rng\": torch.get_rng_state(), \"epoch\": ep,\r\n                    \"best\": best, \"best_ep\": best_ep, \"best_state\": best_state, \"hist\": hist}, st.with_suffix(\".tmp\"))\r\n        os.replace(st.with_suffix(\".tmp\"), st)\r\n        log(f\"  [{cfg.name} s{seed}] ep{ep:02d} loss {h['train_loss']:.3f} valF1 {h['val_macro_f1']:.3f} DCrec {h['val_dc_recall']:.2f} \"\r\n            f\"PLrec {h['val_pl_recall']:.2f} ({h['sec']}s)\")\r\n    model.load_state_dict(best_state)\r\n    prob = predict(torch, model, Xva, device)\r\n    ck = out / \"best.pth\"\r\n    torch.save(best_state, ck)\r\n    sha = sha256_file(ck)\r\n    res = {\"experiment\": cfg.name, \"seed\": seed, \"config\": asdict(cfg), \"train_counts\": dict(zip(CLASSES, counts.tolist())),\r\n           \"train_partial_caries\": int((ytr == PARTIAL_CARIES).sum()),\r\n           \"class_weights\": dict(zip(CLASSES, [round(float(v), 4) for v in w])) if w is not None else None,\r\n           \"sampler_exposure\": sampler_exposure(ytr) if cfg.sampler != \"none\" else None,\r\n           \"best_epoch\": best_ep, \"val_gt\": report(yva, prob.argmax(1), prob), \"history\": hist,\r\n           \"checkpoint\": str(ck), \"checkpoint_sha256\": sha, \"gpu\": gpu_status(torch), \"software\": software(),\r\n           \"split_note\": \"fit on combined_train; epoch selection on combined_validation (= project val); test not used\"}\r\n    np.save(out / \"val_prob.npy\", prob)\r\n    sidecar = {\"task\": \"classification\", \"architecture\": \"resnet50\", \"version\": f\"kaggle_{cfg.name}_s{seed}\", \"dental_specific\": True,\r\n               \"candidate\": True, \"promoted\": False, \"class_names\": CLASSES, \"num_classes\": len(CLASSES), \"image_size\": CROP_SIZE,\r\n               \"crop_pad\": CROP_PAD, \"normalize\": {\"mean\": list(MEAN), \"std\": list(STD)}, \"label_source\": \"category_id_3 (DENTEX disease)\",\r\n               \"class_counts_train\": res[\"train_counts\"], \"class_weights\": res[\"class_weights\"], \"pretrained\": \"torchvision IMAGENET1K_V2\",\r\n               \"training_config\": asdict(cfg), \"seed\": seed, \"software\": res[\"software\"], \"checkpoint_sha256\": sha[:16],\r\n               \"checkpoint_sha256_full\": sha, \"measured_on\": \"validation split (best epoch by macro-F1)\",\r\n               \"metrics\": {\"classification\": {k: res[\"val_gt\"][k] for k in (\"macro_f1\", \"accuracy\", \"per_class\")}},\r\n               \"trained_by\": \"kaggle/denta_mind_training.ipynb\",\r\n               \"limitations\": [\"Candidate only: not validated for clinical use and not promoted.\",\r\n                               \"Trained only on teeth DENTEX labelled with one of four findings; there is no healthy class.\"]}\r\n    ck.with_suffix(\".json\").write_text(json.dumps(sidecar, indent=1), encoding=\"utf-8\")\r\n    (out / \"result.json\").write_text(json.dumps(res, indent=1), encoding=\"utf-8\")\r\n    st.unlink(missing_ok=True)\r\n    del model, opt\r\n    if device.type == \"cuda\":\r\n        torch.cuda.empty_cache()\r\n    return res\r\n\r\n\r\ndef evaluate_checkpoint(ck_path, Xg, y, Xd=None, mi=None, sources=None):\r\n    \"\"\"GT-box metrics, detector-box metrics on matched findings, end-to-end recall (missed = wrong), per-source metrics.\"\"\"\r\n    import torch\r\n    import torchvision as tv\r\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\r\n    model = new_model(torch, tv, pretrained=False)\r\n    model.load_state_dict(torch.load(ck_path, map_location=\"cpu\", weights_only=True), strict=True)\r\n    model.to(device)\r\n    pg = predict(torch, model, Xg, device)\r\n    out = {\"gt\": report(y, pg.argmax(1), pg)}\r\n    if Xd is not None and len(Xd):\r\n        pd = predict(torch, model, Xd, device)\r\n        out[\"det_matched\"] = report(y[mi], pd.argmax(1), pd)\r\n        e2e = np.full(len(y), -1)\r\n        e2e[mi] = pd.argmax(1)\r\n        out[\"end_to_end\"] = {\"accuracy\": float((e2e == y).mean()),\r\n                             \"recall\": {c: float(((e2e == k) & (y == k)).sum() / max(1, (y == k).sum())) for k, c in enumerate(CLASSES)},\r\n                             \"unmatched_findings\": int(len(y) - len(mi))}\r\n    if sources is not None:\r\n        src = np.asarray(sources)\r\n        out[\"by_source\"] = {s: report(y[src == s], pg.argmax(1)[src == s]) for s in sorted(set(sources))}\r\n    del model\r\n    if device.type == \"cuda\":\r\n        torch.cuda.empty_cache()\r\n    return out\r\n\r\n\r\n# --------------------------------------------------------------------------------------------------------- acceptance\r\ndef _mean(xs):\r\n    return float(np.mean(xs)) if xs else float(\"nan\")\r\n\r\n\r\ndef acceptance(cand: dict, base: dict, setting=\"det_matched\"):\r\n    \"\"\"cand/base: {seed: evaluate_checkpoint(...) on validation}. Applies ACCEPT (pre-registered). Returns dict with pass flags.\"\"\"\r\n    seeds = sorted(set(cand) & set(base))\r\n    if len(seeds) < ACCEPT[\"seed_pairs_min\"]:\r\n        return {\"pass\": False, \"reason\": f\"only {len(seeds)} paired seeds\"}\r\n    s = setting if all(setting in cand[k] for k in seeds) else \"gt\"\r\n    f = lambda d, k: d[k][s][\"macro_f1\"]  # noqa: E731\r\n    diffs = [f(cand, k) - f(base, k) for k in seeds]\r\n    cross = sum(f(cand, a) > f(base, b) for a in cand for b in base)\r\n    pc = lambda d, c, m: _mean([d[k][s][\"per_class\"][c][m] for k in seeds])  # noqa: E731\r\n    c1 = sum(d > 0 for d in diffs) >= ACCEPT[\"seed_pairs_min\"] and _mean(diffs) > ACCEPT[\"gain_min\"] and cross >= ACCEPT[\"cross_pairs_min\"]\r\n    c2 = all(pc(cand, c, \"recall\") >= pc(base, c, \"recall\") for c in MINORITY)\r\n    c3 = all(pc(cand, c, \"f1\") >= pc(base, c, \"f1\") - ACCEPT[\"class_f1_drop_max\"] for c in CLASSES)\r\n    gt = lambda d: _mean([d[k][\"gt\"][\"macro_f1\"] for k in seeds])  # noqa: E731\r\n    c4 = gt(cand) >= gt(base) - ACCEPT[\"gt_macro_f1_drop_max\"]\r\n    c5 = all(pc(cand, c, \"precision\") >= pc(base, c, \"precision\") - ACCEPT[\"minority_precision_drop_max\"] for c in MINORITY)\r\n    return {\"setting\": s, \"seeds\": seeds, \"mean_gain_macro_f1\": _mean(diffs), \"seed_pairs_better\": int(sum(d > 0 for d in diffs)),\r\n            \"cross_pairs_better\": int(cross), \"cross_pairs_total\": len(cand) * len(base),\r\n            \"minority_recall_delta\": {c: pc(cand, c, \"recall\") - pc(base, c, \"recall\") for c in MINORITY},\r\n            \"minority_precision_delta\": {c: pc(cand, c, \"precision\") - pc(base, c, \"precision\") for c in MINORITY},\r\n            \"class_f1_delta\": {c: pc(cand, c, \"f1\") - pc(base, c, \"f1\") for c in CLASSES},\r\n            \"gt_macro_f1_delta\": gt(cand) - gt(base),\r\n            \"criteria\": {\"1_macro_f1_gain\": bool(c1), \"2_minority_recall_not_lower\": bool(c2), \"3_no_class_f1_drop_gt_0.05\": bool(c3),\r\n                         \"4_gt_macro_f1_drop_le_0.02\": bool(c4), \"5_minority_precision_drop_le_0.10\": bool(c5)},\r\n            \"pass\": bool(c1 and c2 and c3 and c4 and c5)}\r\n\r\n\r\ndef file_hashes(paths):\r\n    return {str(p): sha256_file(p) for p in paths if Path(p).is_file()}\r\n\r\n\r\ndef short_hash(obj) -> str:\r\n    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:16]\r\n\r\n\r\n\r\n# -------------------------------------------------------------------------------------------- process-isolated training\r\ndef save_arrays(path, data):\r\n    \"\"\"Write a {\"X\", \"y\"} split once, for the training subprocesses (np.savez, uncompressed).\"\"\"\r\n    if not Path(path).is_file():\r\n        np.savez(path, X=data[\"X\"], y=data[\"y\"])\r\n    return str(path)\r\n\r\n\r\ndef train_run_isolated(cfg: ExpConfig, seed: int, train_npz, val_npz, out_dir, log=print, attempts=3, timeout_s=3 * 3600):\r\n    \"\"\"Run ``train_run`` in a fresh Python process. A CUDA fault (e.g. 'illegal memory access') poisons the whole process's GPU\r\n    context, so isolating every run keeps one fault from killing the remaining experiments. A failed attempt is retried and\r\n    resumes from its last completed epoch (state.pt). Returns the result dict, or {\"failed\": True, ...} after ``attempts``.\"\"\"\r\n    import subprocess\r\n    out = Path(out_dir) / f\"{cfg.name}_s{seed}\"\r\n    if (out / \"result.json\").is_file():\r\n        log(f\"[{cfg.name} s{seed}] already finished - skipped\")\r\n        return json.loads((out / \"result.json\").read_text(encoding=\"utf-8\"))\r\n    spec = json.dumps({\"cfg\": asdict(cfg), \"seed\": seed, \"train\": str(train_npz), \"val\": str(val_npz), \"out\": str(out_dir)})\r\n    errors = []\r\n    for attempt in range(1, attempts + 1):\r\n        p = subprocess.Popen([sys.executable, \"-u\", str(Path(__file__).resolve()), \"--worker\", spec],\r\n                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)\r\n        tail = []\r\n        try:\r\n            for line in p.stdout:\r\n                line = line.rstrip()\r\n                tail = (tail + [line])[-40:]\r\n                if line.startswith(\"  [\") or \"resumed\" in line or \"Error\" in line:\r\n                    log(line)\r\n            rc = p.wait(timeout=timeout_s)\r\n        except subprocess.TimeoutExpired:\r\n            p.kill()\r\n            rc = \"timeout\"\r\n        if rc == 0 and (out / \"result.json\").is_file():\r\n            if attempt > 1:\r\n                log(f\"[{cfg.name} s{seed}] succeeded on attempt {attempt}\")\r\n            return json.loads((out / \"result.json\").read_text(encoding=\"utf-8\"))\r\n        errors.append({\"attempt\": attempt, \"returncode\": rc, \"tail\": tail[-15:]})\r\n        log(f\"[{cfg.name} s{seed}] attempt {attempt} FAILED (rc={rc}): \" + \" | \".join(tail[-4:]))\r\n    out.mkdir(parents=True, exist_ok=True)\r\n    (out / \"failure.json\").write_text(json.dumps(errors, indent=1), encoding=\"utf-8\")\r\n    return {\"failed\": True, \"experiment\": cfg.name, \"seed\": seed, \"errors\": errors}\r\n\r\n\r\ndef _worker(spec_json):\r\n    spec = json.loads(spec_json)\r\n    cfg = ExpConfig(**spec[\"cfg\"])\r\n    load = lambda f: (lambda z: {\"X\": z[\"X\"], \"y\": z[\"y\"]})(np.load(f))  # noqa: E731\r\n    train_run(cfg, spec[\"seed\"], load(spec[\"train\"]), load(spec[\"val\"]), spec[\"out\"], log=print)\r\n\r\n\r\nif __name__ == \"__main__\" and len(sys.argv) == 3 and sys.argv[1] == \"--worker\":\r\n    _worker(sys.argv[2])\r\n", "kaggle/cv5_lib.py": "\"\"\"Grouped 5-fold cross-validation of the BASE finding classifier. Protocol: reports/cv5_protocol.md (pre-registered 2026-10-24).\n\nReuses kaggle/dentamind_kaggle.py (crops, BASE recipe, isolated training, prediction, metrics) and scripts/dataset/make_cv_folds.py\n(group assignment). This module only adds: fold/leakage gates, the per-fold plan (fit / inner-validation / held-out), a detector that\nreproduces the production inference path, out-of-fold aggregation, a radiograph-level bootstrap and the Caries vs Deep Caries analysis.\n\"\"\"\nfrom __future__ import annotations\n\nimport csv\nimport dataclasses\nimport hashlib\nimport json\nimport sys\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nimport numpy as np\n\nHERE = Path(__file__).resolve().parent\nfor p in (HERE, HERE.parent / \"scripts\" / \"dataset\"):\n    if p.is_dir() and str(p) not in sys.path:\n        sys.path.insert(0, str(p))\nimport dentamind_kaggle as DK  # noqa: E402\nimport make_cv_folds as MCF  # noqa: E402\nfrom dsutil import CLASSES  # noqa: E402\n\nFOLD_FILE_SHA256 = \"9b767a4822844ccf0c4bf33d299e6e37420d1458a38035af0c3a3cb9107b47a6\"   # protocol section 2\nK = 5\nSEEDS = [42, 43, 44, 45, 46]\nINNER_K = 8\nPRIMARY = \"dentex_train_disease\"\nSECONDARY = \"dentex_official_val\"\nPOOL_SPLITS = (\"train\", \"validation\")\nEXCLUDED_SPLITS = (\"calibration\", \"test\")\nBOOT_N, BOOT_SEED = 2000, 0\nDET_IOU_MEDIAN_MIN = 0.98\n\n\nclass CVError(RuntimeError):\n    pass\n\n\ndef sha256_file(p) -> str:\n    h = hashlib.sha256()\n    with open(p, \"rb\") as fh:\n        for b in iter(lambda: fh.read(1 << 20), b\"\"):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef run_name(fold: int) -> str:\n    return f\"CV5_BASE_f{fold}\"\n\n\ndef fold_config(fold: int) -> DK.ExpConfig:\n    \"\"\"The BASE recipe, unchanged except for the run name (protocol section 3).\"\"\"\n    return dataclasses.replace(DK.EXPERIMENTS[\"BASE\"], name=run_name(fold),\n                               note=f\"CV5 fold {fold}: BASE recipe (C0r), fit on 4 folds minus inner validation; reports/cv5_protocol.md\")\n\n\n# ------------------------------------------------------------------------------------------------------------- folds (gate 1)\ndef load_folds(path, expected_sha=FOLD_FILE_SHA256) -> dict:\n    \"\"\"{radiograph_group: fold} from the approved fold file; refuses any other file.\"\"\"\n    got = sha256_file(path)\n    if expected_sha and got != expected_sha:\n        raise CVError(f\"fold file sha256 {got[:16]}... is not the approved {expected_sha[:16]}...\")\n    fold_of = {}\n    with open(path, encoding=\"utf-8\") as fh:\n        for r in csv.DictReader(fh):\n            g, f = r[\"radiograph_group\"], int(r[\"fold\"])\n            if g in fold_of:\n                raise CVError(f\"group {g} listed twice in the fold file\")\n            if not 0 <= f < K:\n                raise CVError(f\"group {g}: fold {f} outside 0..{K - 1}\")\n            fold_of[g] = f\n    return fold_of\n\n\ndef check_folds(fold_of: dict, rows_by_split: dict) -> dict:\n    \"\"\"Gate 1: every pool group exactly once; no calibration/test group in the pool; no image (SHA-256) in two folds.\"\"\"\n    pool = [r for s in POOL_SPLITS for r in rows_by_split[s]]\n    pool_groups = {r[\"radiograph_group\"] for r in pool}\n    excluded = {r[\"radiograph_group\"] for s in EXCLUDED_SPLITS for r in rows_by_split.get(s, [])}\n    excluded_sha = {r[\"image_sha256\"] for s in EXCLUDED_SPLITS for r in rows_by_split.get(s, [])}\n    problems = []\n    if set(fold_of) != pool_groups:\n        problems.append(f\"fold file groups != pool groups (missing {len(pool_groups - set(fold_of))}, extra {len(set(fold_of) - pool_groups)})\")\n    if set(fold_of) & excluded:\n        problems.append(f\"{len(set(fold_of) & excluded)} calibration/test groups in the fold file\")\n    sha_folds = defaultdict(set)\n    for r in pool:\n        sha_folds[r[\"image_sha256\"]].add(fold_of.get(r[\"radiograph_group\"]))\n        if r[\"image_sha256\"] in excluded_sha:\n            problems.append(f\"pool image {r['image_path']} is byte-identical to a calibration/test image\")\n    multi = [s for s, f in sha_folds.items() if len(f) > 1]\n    if multi:\n        problems.append(f\"{len(multi)} images in two folds\")\n    if problems:\n        raise CVError(\"fold gate failed: \" + \"; \".join(problems[:5]))\n    per_fold = defaultdict(Counter)\n    for r in pool:\n        per_fold[fold_of[r[\"radiograph_group\"]]][(r[\"source_dataset\"], r[\"target_class\"])] += 1\n    return {\"pool_groups\": len(pool_groups), \"pool_findings\": len(pool), \"excluded_groups_checked\": len(excluded),\n            \"radiographs_per_fold\": dict(sorted(Counter(fold_of.values()).items())),\n            \"findings_per_fold\": {f: {f\"{s}:{c}\": n for (s, c), n in sorted(cnt.items())} for f, cnt in sorted(per_fold.items())}}\n\n\n# ------------------------------------------------------------------------------------------------------------- plan (gate 2)\ndef fold_plan(pool_rows: list, fold_of: dict, fold: int) -> dict:\n    \"\"\"Protocol section 3. Returns row lists: fit / inner (BASE data rule: primary source only), heldout (primary), heldout_secondary.\"\"\"\n    heldout = [r for r in pool_rows if fold_of[r[\"radiograph_group\"]] == fold]\n    train_side = [r for r in pool_rows if fold_of[r[\"radiograph_group\"]] != fold and r[\"source_dataset\"] == PRIMARY]\n    inner_of = MCF.assign_folds(train_side, k=INNER_K, seed=42 + fold, with_impacted=True)\n    inner = [r for r in train_side if inner_of[r[\"radiograph_group\"]] == 0]\n    fit = [r for r in train_side if inner_of[r[\"radiograph_group\"]] != 0]\n    return {\"fold\": fold, \"fit\": fit, \"inner\": inner,\n            \"heldout\": [r for r in heldout if r[\"source_dataset\"] == PRIMARY],\n            \"heldout_secondary\": [r for r in heldout if r[\"source_dataset\"] == SECONDARY]}\n\n\ndef check_plan(plan: dict, fold_of: dict) -> dict:\n    \"\"\"Gate 2: fit, inner and held-out are pairwise disjoint by group and by image; held-out = exactly fold k.\"\"\"\n    k = plan[\"fold\"]\n    parts = {n: plan[n] for n in (\"fit\", \"inner\", \"heldout\", \"heldout_secondary\")}\n    g = {n: {r[\"radiograph_group\"] for r in rs} for n, rs in parts.items()}\n    s = {n: {r[\"image_sha256\"] for r in rs} for n, rs in parts.items()}\n    names = list(parts)\n    for i, a in enumerate(names):\n        for b in names[i + 1:]:\n            if g[a] & g[b] or s[a] & s[b]:\n                raise CVError(f\"fold {k}: {a} and {b} share {len(g[a] & g[b])} groups / {len(s[a] & s[b])} images\")\n    if any(fold_of[x] == k for x in g[\"fit\"] | g[\"inner\"]):\n        raise CVError(f\"fold {k}: a held-out radiograph is on the training side\")\n    if any(fold_of[x] != k for x in g[\"heldout\"] | g[\"heldout_secondary\"]):\n        raise CVError(f\"fold {k}: held-out set contains another fold\")\n    if {r[\"source_dataset\"] for r in plan[\"fit\"] + plan[\"inner\"]} - {PRIMARY}:\n        raise CVError(f\"fold {k}: non-BASE source in the fitting data\")\n    if not plan[\"fit\"] or not plan[\"inner\"] or not plan[\"heldout\"]:\n        raise CVError(f\"fold {k}: an empty part\")\n    return {n: {\"radiographs\": len(g[n]), \"findings\": len(parts[n]),\n                \"classes\": dict(sorted(Counter(r[\"target_class\"] for r in parts[n]).items()))} for n in names}\n\n\n# ------------------------------------------------------------------------------------------------------------- detector\ndef load_gray01(path) -> np.ndarray:\n    \"\"\"Same as dentexmodel.data.preprocess.load_gray (production input of the detector).\"\"\"\n    from PIL import Image\n    with Image.open(path) as im:\n        im.load()\n        if im.mode in (\"I;16\", \"I;16L\", \"I;16B\", \"I\"):\n            arr = np.asarray(im, dtype=np.float32)\n            top = float(arr.max()) or 1.0\n            return np.clip(arr / (65535.0 if top > 255 else 255.0), 0, 1).astype(np.float32)\n        if im.mode in (\"RGBA\", \"LA\", \"P\"):\n            im = im.convert(\"RGBA\").convert(\"L\") if im.mode != \"LA\" else im.convert(\"L\")\n        else:\n            im = im.convert(\"L\")\n        return np.asarray(im, dtype=np.float32) / 255.0\n\n\nclass ProductionDetector:\n    \"\"\"dentexmodel.inference.torch_models._run_detector for dentex_teeth_detection_v1: grayscale replicated to 3 channels, float [0,1],\n    autocast on CUDA (inference.mixed_precision default True), score >= DK.DET_SCORE (0.3).\"\"\"\n\n    def __init__(self, ckpt, device=None):\n        import torch\n        import torchvision as tv\n        self.torch = torch\n        self.device = device or torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n        m = tv.models.detection.fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None, num_classes=2)\n        m.load_state_dict(torch.load(ckpt, map_location=\"cpu\", weights_only=True), strict=True)\n        self.model = m.to(self.device).eval()\n\n    def boxes(self, path) -> list:\n        t = self.torch\n        g = load_gray01(path)\n        rgb01 = np.repeat(g[..., None], 3, axis=2)\n        x = t.from_numpy(np.ascontiguousarray(rgb01.transpose(2, 0, 1))).float().to(self.device)\n        with t.no_grad(), t.autocast(device_type=self.device.type, enabled=self.device.type == \"cuda\"):\n            o = self.model([x])[0]\n        return [[float(v) for v in b] + [float(s)] for b, s in zip(o[\"boxes\"].float().cpu().tolist(), o[\"scores\"].float().cpu().tolist())\n                if s >= DK.DET_SCORE]\n\n\ndef gt_boxes(rows) -> list:\n    out = []\n    for r in rows:\n        x, y, w, h = json.loads(r[\"bbox_xywh\"])\n        out.append([x, y, x + w, y + h])\n    return out\n\n\ndef compare_detector(rows, new: dict, cached: dict, key=lambda r: r[\"project_image_id\"]) -> dict:\n    \"\"\"Gate 3 statistic: per image, match GT findings to cached and to regenerated boxes (DK.match); the matched finding sets must be\n    identical, and corresponding boxes must agree (median IoU).\"\"\"\n    by_img = defaultdict(list)\n    for r in rows:\n        by_img[key(r)].append(r)\n    same, diff, ious = 0, [], []\n    for kk, rs in by_img.items():\n        gts = gt_boxes(rs)\n        cb = [d[:4] for d in cached.get(kk, []) if d[4] >= DK.DET_SCORE]\n        nb = [d[:4] for d in new.get(kk, []) if d[4] >= DK.DET_SCORE]\n        mc, mn = DK.match(gts, cb), DK.match(gts, nb)\n        if set(mc) == set(mn):\n            same += len(rs)\n        else:\n            diff.append({\"image\": kk, \"only_cached\": sorted(set(mc) - set(mn)), \"only_new\": sorted(set(mn) - set(mc))})\n        for i in set(mc) & set(mn):\n            ious.append(DK.box_iou(cb[mc[i][0]], nb[mn[i][0]]))\n    med = float(np.median(ious)) if ious else 0.0\n    return {\"images\": len(by_img), \"findings\": len(rows), \"findings_same_match\": same, \"images_with_different_match\": diff,\n            \"median_box_iou\": med, \"pass\": (not diff) and med >= DET_IOU_MEDIAN_MIN}\n\n\n# ------------------------------------------------------------------------------------------------------------- evaluation\ndef macro_f1(y, pred, n=len(CLASSES)) -> float:\n    \"\"\"Same definition as DK.report (vectorised for the bootstrap).\"\"\"\n    cm = np.bincount(np.asarray(y) * n + np.asarray(pred), minlength=n * n).reshape(n, n)\n    tp, pp, sup = np.diag(cm), cm.sum(0), cm.sum(1)\n    prec = np.divide(tp, pp, out=np.zeros(n), where=pp > 0)\n    rec = np.divide(tp, sup, out=np.zeros(n), where=sup > 0)\n    return float(np.divide(2 * prec * rec, prec + rec, out=np.zeros(n), where=(prec + rec) > 0).mean())\n\n\ndef check_oof(oof: dict, n_findings: int, seeds=SEEDS) -> None:\n    \"\"\"Gate 5: oof[seed]['gt'] is (N, C) with every row filled exactly once (no NaN), for every seed.\"\"\"\n    for s in seeds:\n        if s not in oof:\n            raise CVError(f\"seed {s}: no out-of-fold predictions\")\n        p = oof[s][\"gt\"]\n        if p.shape != (n_findings, len(CLASSES)) or np.isnan(p).any():\n            raise CVError(f\"seed {s}: out-of-fold GT predictions incomplete ({int(np.isnan(p).any(1).sum())} findings missing)\")\n        if (oof[s][\"filled\"] != 1).any():\n            raise CVError(f\"seed {s}: {int((oof[s]['filled'] != 1).sum())} findings predicted 0 or >1 times\")\n\n\ndef pooled_reports(y, oof: dict, matched: np.ndarray, seeds=SEEDS, subset=None) -> dict:\n    \"\"\"Per seed: DK.report on pooled out-of-fold predictions; detector setting on matched findings only; end-to-end recall.\"\"\"\n    idx = np.arange(len(y)) if subset is None else np.asarray(subset)\n    out = {}\n    for s in seeds:\n        pg, pdet = oof[s][\"gt\"][idx], oof[s][\"det\"][idx]\n        m = matched[idx]\n        r = {\"gt\": DK.report(y[idx], pg.argmax(1), pg)}\n        if m.any():\n            r[\"det_matched\"] = DK.report(y[idx][m], pdet[m].argmax(1), pdet[m])\n        e2e = np.where(m, np.nan_to_num(pdet, nan=0).argmax(1), -1)\n        r[\"end_to_end\"] = {\"accuracy\": float((e2e == y[idx]).mean()), \"unmatched_findings\": int((~m).sum()),\n                           \"recall\": {c: float(((e2e == k) & (y[idx] == k)).sum() / max(1, (y[idx] == k).sum())) for k, c in enumerate(CLASSES)}}\n        out[s] = r\n    return out\n\n\ndef cluster_bootstrap(y, preds_by_seed: list, groups, valid=None, n_boot=BOOT_N, seed=BOOT_SEED) -> dict:\n    \"\"\"95 % percentile interval of the seed-mean macro-F1, resampling RADIOGRAPHS (all their findings) with replacement.\n    valid: boolean mask of findings that count (e.g. detector-matched); invalid findings are dropped after resampling.\"\"\"\n    y = np.asarray(y)\n    groups = np.asarray(groups)\n    valid = np.ones(len(y), bool) if valid is None else np.asarray(valid)\n    uniq = np.unique(groups)\n    members = {g: np.flatnonzero(groups == g) for g in uniq}\n    rng = np.random.default_rng(seed)\n    stats = np.empty(n_boot)\n    for b in range(n_boot):\n        pick = rng.choice(uniq, size=len(uniq), replace=True)\n        i = np.concatenate([members[g] for g in pick])\n        i = i[valid[i]]\n        stats[b] = np.mean([macro_f1(y[i], np.asarray(p)[i]) for p in preds_by_seed])\n    point = float(np.mean([macro_f1(y[valid], np.asarray(p)[valid]) for p in preds_by_seed]))\n    return {\"unit\": \"radiograph\", \"n_radiographs\": int(len(uniq)), \"n_boot\": n_boot, \"seed\": seed, \"point\": point,\n            \"ci95\": [float(np.percentile(stats, 2.5)), float(np.percentile(stats, 97.5))], \"boot_sd\": float(stats.std(ddof=1))}\n\n\ndef mean_sd(values) -> dict:\n    v = [float(x) for x in values]\n    return {\"values\": v, \"mean\": float(np.mean(v)), \"sd\": float(np.std(v, ddof=1)) if len(v) > 1 else 0.0}\n\n\ndef tooth_type(fdi) -> str:\n    try:\n        d = int(str(fdi)[-1])\n    except (TypeError, ValueError):\n        return \"unknown\"\n    return {1: \"incisor\", 2: \"incisor\", 3: \"canine\", 4: \"premolar\", 5: \"premolar\"}.get(d, \"molar\" if d in (6, 7, 8) else \"unknown\")\n\n\ndef caries_confusion(rows, oof: dict, matched: np.ndarray, seeds=SEEDS) -> dict:\n    \"\"\"Exploratory (protocol section 6). For every Caries / Deep Caries finding: how many seeds predicted the other caries class,\n    per setting. Returns per-finding records plus summaries by tooth type and box-area quartile. Labels are not judged.\"\"\"\n    C, DC = CLASSES.index(\"Caries\"), CLASSES.index(\"Deep Caries\")\n    y = np.array([int(r[\"class_index\"]) for r in rows])\n    sel = np.flatnonzero((y == C) | (y == DC))\n    other = np.where(y == C, DC, C)\n    n_gt = np.zeros(len(y), int)\n    n_det = np.zeros(len(y), int)\n    for s in seeds:\n        n_gt += oof[s][\"gt\"].argmax(1) == other\n        n_det += np.where(matched, np.nan_to_num(oof[s][\"det\"], nan=0).argmax(1) == other, False)\n    area = np.array([np.prod(json.loads(r[\"bbox_xywh\"])[2:]) for r in rows], float)\n    q = np.quantile(area[sel], [0.25, 0.5, 0.75])\n    quart = np.searchsorted(q, area, side=\"right\") + 1\n    recs = []\n    for i in sel:\n        r = rows[i]\n        recs.append({\"sample_id\": r[\"sample_id\"], \"image_path\": r[\"image_path\"], \"radiograph_group\": r[\"radiograph_group\"],\n                     \"fdi_tooth\": r[\"fdi_tooth\"], \"tooth_type\": tooth_type(r[\"fdi_tooth\"]), \"label\": r[\"target_class\"],\n                     \"original_label\": r[\"original_label\"], \"box_area_px\": int(area[i]), \"box_area_quartile\": int(quart[i]),\n                     \"detector_matched\": bool(matched[i]), \"seeds_other_caries_gt\": int(n_gt[i]),\n                     \"seeds_other_caries_det\": int(n_det[i]) if matched[i] else \"\"})\n    def summary(key):\n        out = {}\n        for lab in (\"Caries\", \"Deep Caries\"):\n            for v in sorted({rr[key] for rr in recs}):\n                grp = [rr for rr in recs if rr[\"label\"] == lab and rr[key] == v]\n                if grp:\n                    out[f\"{lab} | {key}={v}\"] = {\"n\": len(grp), \"mean_share_of_seeds_other_caries_gt\":\n                                                 float(np.mean([rr[\"seeds_other_caries_gt\"] for rr in grp]) / len(seeds)),\n                                                 \"confused_by_all_seeds_gt\": sum(rr[\"seeds_other_caries_gt\"] == len(seeds) for rr in grp)}\n        return out\n    return {\"records\": recs, \"box_area_quartile_edges_px\": [float(v) for v in q],\n            \"by_tooth_type\": summary(\"tooth_type\"), \"by_box_area_quartile\": summary(\"box_area_quartile\"),\n            \"confused_by_all_seeds_gt\": {lab: sum(rr[\"label\"] == lab and rr[\"seeds_other_caries_gt\"] == len(seeds) for rr in recs)\n                                         for lab in (\"Caries\", \"Deep Caries\")},\n            \"n\": {lab: sum(rr[\"label\"] == lab for rr in recs) for lab in (\"Caries\", \"Deep Caries\")}}\n", "kaggle/cv5_accept.py": "\"\"\"CV5 acceptance rule for candidates vs the BASE reference. Protocol: reports/cv5_acceptance_protocol.md (revision 3, approved 2026-10-25).\r\n\r\nImplements, and only implements, what the protocol specifies:\r\n* paired radiograph-level bootstrap (Appendix C) and the criteria P-A..P-D, S1..S7, H (section 5);\r\n* validity gates: fold-file check and gate R (section 4.3), gate V6 (section 3.1);\r\n* run classification and collapse check (section 9), lifetime budget (section 10.1), winner rule (section 10.2);\r\n* the V-a held-out fold-0 report from stored predictions (section 10.3) and the per-run training manifest it requires.\r\n\r\nIt trains nothing and runs no inference. Thresholds are module constants copied from the protocol; changing one is a protocol amendment.\r\n\"\"\"\r\nfrom __future__ import annotations\r\n\r\nimport hashlib\r\nimport json\r\nfrom datetime import datetime, timezone\r\nfrom pathlib import Path\r\n\r\nimport numpy as np\r\n\r\nimport cv5_lib as CV\r\nimport dentamind_kaggle as DK\r\nfrom dsutil import CLASSES\r\n\r\nPROTOCOL = \"reports/cv5_acceptance_protocol.md (revision 3, approved 2026-10-25)\"\r\nDELTA = 0.027                                   # P-A, operator >=\r\nK_LIFETIME = 15                                 # U6: Bonferroni over the fixed lifetime budget\r\nB, BOOT_SEED = 20_000, 0                        # Appendix C\r\nPB_LEVEL = 1 - 0.05 / K_LIFETIME\r\nH_LEVEL = 0.95                                  # U7: hard-harm and safeguard CIs, unadjusted\r\nSEEDS_MIN_WINS = FOLDS_MIN_WINS = 4\r\nDC, PL = CLASSES.index(\"Deep Caries\"), CLASSES.index(\"Periapical Lesion\")\r\n# (name, setting, class index or None for macro, metric, margin)\r\nSAFEGUARDS = [(\"S1\", \"det\", DC, \"recall\", 0.03), (\"S2\", \"det\", PL, \"recall\", 0.05),\r\n              (\"S3\", \"det\", DC, \"precision\", 0.04), (\"S4\", \"det\", PL, \"precision\", 0.04),\r\n              (\"S5a\", \"det\", CLASSES.index(\"Caries\"), \"f1\", 0.02), (\"S5b\", \"det\", DC, \"f1\", 0.03),\r\n              (\"S5c\", \"det\", CLASSES.index(\"Impacted\"), \"f1\", 0.02), (\"S5d\", \"det\", PL, \"f1\", 0.03),\r\n              (\"S6\", \"gt\", None, \"macro_f1\", 0.02),\r\n              (\"S7a\", \"e2e\", DC, \"recall\", 0.03), (\"S7b\", \"e2e\", PL, \"recall\", 0.05)]\r\nHARM = [(\"H_DC_recall\", DC, \"recall\"), (\"H_PL_recall\", PL, \"recall\"),\r\n        (\"H_DC_precision\", DC, \"precision\"), (\"H_PL_precision\", PL, \"precision\")]\r\nR2_TOL = 1e-4                                   # gate R, max |probability difference|\r\nCOLLAPSE_SHARE = 0.99                           # section 9\r\nMAX_ATTEMPTS = 3\r\nFAMILY_CAP = {\"F1\": 3, \"F2\": 3, \"F3\": 3, \"F4\": 3, \"F5\": 1, \"combination\": 2}\r\nFINAL_FOLD, FINAL_SEED = 0, 42                  # V-a\r\nDISCLOSURE = (\"Held-out fold 0 was part of the population on which all confirmatory candidates were compared and the winner was \"\r\n              \"selected. This report is independent of the checkpoint's training data but not independent confirmation of the \"\r\n              \"selected winner.\")\r\nNC = len(CLASSES)\r\nUNMATCHED = NC                                  # extra predicted column for end-to-end (unmatched = wrong)\r\n\r\n\r\nclass AcceptError(RuntimeError):\r\n    pass\r\n\r\n\r\n# ------------------------------------------------------------------------------------------------------------- predictions\r\ndef labels(arm: dict) -> dict:\r\n    \"\"\"arm = {seed: {\"gt\": (N, C) probs, \"det\": (N, C) probs, NaN rows = no detector match}} -> predicted classes per setting.\r\n    det: argmax on matched rows (-1 elsewhere); e2e: det argmax, unmatched -> UNMATCHED.\"\"\"\r\n    out = {}\r\n    for s, p in arm.items():\r\n        det = np.asarray(p[\"det\"], float)\r\n        ok = ~np.isnan(det).any(1)\r\n        dl = np.where(ok, np.nan_to_num(det, nan=-1).argmax(1), -1)\r\n        out[s] = {\"gt\": np.asarray(p[\"gt\"], float).argmax(1), \"det\": dl, \"e2e\": np.where(ok, dl, UNMATCHED), \"matched\": ok}\r\n    return out\r\n\r\n\r\ndef metrics_from_confusion(cm: np.ndarray) -> dict:\r\n    \"\"\"cm: (..., C, C+1) counts, rows = true class, last column = unmatched. Undefined precision / recall / F1 count as 0.\"\"\"\r\n    tp = np.diagonal(cm[..., :NC], axis1=-2, axis2=-1).astype(float)\r\n    pp = cm[..., :NC].sum(-2).astype(float)\r\n    sup = cm.sum(-1).astype(float)\r\n    prec = np.divide(tp, pp, out=np.zeros_like(tp), where=pp > 0)\r\n    rec = np.divide(tp, sup, out=np.zeros_like(tp), where=sup > 0)\r\n    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(tp), where=(prec + rec) > 0)\r\n    return {\"precision\": prec, \"recall\": rec, \"f1\": f1, \"macro_f1\": f1.mean(-1), \"support\": sup}\r\n\r\n\r\ndef _group_confusion(y, pred, mask, gidx, n_groups) -> np.ndarray:\r\n    \"\"\"(G, C*(C+1)) confusion counts per radiograph group for the findings in mask.\"\"\"\r\n    out = np.zeros((n_groups, NC * (NC + 1)), np.int64)\r\n    cell = y * (NC + 1) + pred\r\n    np.add.at(out, (gidx[mask], cell[mask]), 1)\r\n    return out\r\n\r\n\r\ndef _weights(rng, n_groups, n_boot, chunk=2000):\r\n    \"\"\"Yields (draws, counts): draws = rng.choice(G, size=(n, G), replace=True) per chunk; counts[b, g] = times group g was drawn.\"\"\"\r\n    for start in range(0, n_boot, chunk):\r\n        n = min(chunk, n_boot - start)\r\n        d = rng.choice(n_groups, size=(n, n_groups), replace=True)\r\n        c = np.bincount((d + n_groups * np.arange(n)[:, None]).ravel(), minlength=n * n_groups).reshape(n, n_groups)\r\n        yield d, c\r\n\r\n\r\n# ------------------------------------------------------------------------------------------------------------- comparison\r\ndef compare(y, groups, folds, ref: dict, cand: dict, *, n_boot=B, seed=BOOT_SEED, ensemble=False) -> dict:\r\n    \"\"\"Section 5 + Appendix C. y: true class per primary finding; groups: radiograph group; folds: held-out fold.\r\n    ref / cand: arms as in labels(). The detector-matched set comes from the reference (same detector boxes for both arms).\"\"\"\r\n    y, groups, folds = np.asarray(y), np.asarray(groups), np.asarray(folds)\r\n    R, C = labels(ref), labels(cand)\r\n    rs, cs = sorted(R), sorted(C)\r\n    matched = R[rs[0]][\"matched\"]\r\n    for arm in (R, C):\r\n        for s in arm:\r\n            if not np.array_equal(arm[s][\"matched\"], matched):\r\n                raise AcceptError(f\"seed {s}: detector-matched set differs from the reference (same detector boxes required)\")\r\n    if not ensemble and rs != cs:\r\n        raise AcceptError(f\"seed sets differ: reference {rs}, candidate {cs}\")\r\n    uniq, gidx = np.unique(groups, return_inverse=True)\r\n    G = len(uniq)\r\n    masks = {\"det\": matched, \"gt\": np.ones(len(y), bool), \"e2e\": np.ones(len(y), bool)}\r\n\r\n    gc = {(a, s, st): _group_confusion(y, arm[s][st], masks[st], gidx, G)\r\n          for a, arm in ((\"ref\", R), (\"cand\", C)) for s in arm for st in masks}\r\n\r\n    def seed_mean(a, seeds, st, w):            # w: (n, G) group weights -> metrics of the seed-mean arm\r\n        ms = [metrics_from_confusion((w @ gc[(a, s, st)]).reshape(-1, NC, NC + 1)) for s in seeds]\r\n        return {k: np.mean([m[k] for m in ms], 0) for k in (\"precision\", \"recall\", \"f1\", \"macro_f1\")}, ms\r\n\r\n    def quantity(m, cls, metric):\r\n        return m[metric] if cls is None else m[metric][..., cls]\r\n\r\n    ones = np.ones((1, G))\r\n    point = {st: {\"ref\": seed_mean(\"ref\", rs, st, ones)[0], \"cand\": seed_mean(\"cand\", cs, st, ones)[0]} for st in masks}\r\n\r\n    def pdelta(st, cls, metric):\r\n        return float(quantity(point[st][\"cand\"], cls, metric)[0] - quantity(point[st][\"ref\"], cls, metric)[0])\r\n\r\n    # bootstrap: one generator, the same draws for every metric and setting\r\n    rng = np.random.default_rng(seed)\r\n    qs = [(\"macro_f1\", \"det\", None, \"macro_f1\")] + [(n, st, c, m) for n, st, c, m, _ in SAFEGUARDS] + [(n, \"det\", c, m) for n, c, m in HARM]\r\n    boot = {q[0]: np.empty(n_boot) for q in qs}\r\n    degenerate = {st: [0] * NC for st in masks}\r\n    hsh = hashlib.sha256(np.asarray(uniq).astype(str).astype(\"U\").tobytes())\r\n    b0 = 0\r\n    for d, w in _weights(rng, G, n_boot):\r\n        hsh.update(d.astype(\"<i4\").tobytes())\r\n        mr = {st: seed_mean(\"ref\", rs, st, w) for st in masks}\r\n        mc = {st: seed_mean(\"cand\", cs, st, w)[0] for st in masks}\r\n        for name, st, cls, metric in qs:\r\n            boot[name][b0:b0 + len(w)] = quantity(mc[st], cls, metric) - quantity(mr[st][0], cls, metric)\r\n        for st in masks:\r\n            sup = mr[st][1][0][\"support\"]\r\n            for k in range(NC):\r\n                degenerate[st][k] += int((sup[:, k] == 0).sum())\r\n        b0 += len(w)\r\n\r\n    def ci(name, level):\r\n        a = (1 - level) / 2 * 100\r\n        return [float(np.percentile(boot[name], a)), float(np.percentile(boot[name], 100 - a))]\r\n\r\n    out = {\"protocol\": PROTOCOL, \"n_findings\": int(len(y)), \"n_detector_matched\": int(matched.sum()), \"n_radiographs\": int(G),\r\n           \"reference_seeds\": rs, \"candidate_seeds\": cs, \"ensemble\": bool(ensemble),\r\n           \"bootstrap\": {\"generator\": f\"numpy.random.default_rng({seed})\", \"B\": int(n_boot), \"k\": K_LIFETIME,\r\n                         \"draw_call\": \"rng.choice(G, size=(2000, G), replace=True) per chunk of 2000 resamples\",\r\n                         \"draws_sha256\": hsh.hexdigest(), \"degenerate_resamples_per_class\": degenerate}}\r\n    crit = {}\r\n    d_macro = pdelta(\"det\", None, \"macro_f1\")\r\n    crit[\"P-A\"] = {\"delta\": d_macro, \"threshold\": DELTA, \"pass\": d_macro >= DELTA}\r\n    lo, hi = ci(\"macro_f1\", PB_LEVEL)\r\n    crit[\"P-B\"] = {\"level\": PB_LEVEL, \"ci\": [lo, hi], \"pass\": lo > 0}\r\n    if ensemble:\r\n        crit[\"P-C\"] = {\"applicable\": False, \"pass\": True, \"note\": \"ensemble: P-C not applicable (section 6.1)\"}\r\n    else:\r\n        per_seed = {}\r\n        for s in rs:\r\n            per_seed[str(s)] = float(metrics_from_confusion((ones @ gc[(\"cand\", s, \"det\")]).reshape(-1, NC, NC + 1))[\"macro_f1\"][0]\r\n                                     - metrics_from_confusion((ones @ gc[(\"ref\", s, \"det\")]).reshape(-1, NC, NC + 1))[\"macro_f1\"][0])\r\n        wins = sum(v > 0 for v in per_seed.values())\r\n        crit[\"P-C\"] = {\"applicable\": True, \"per_seed_delta\": per_seed, \"wins\": wins, \"pass\": wins >= SEEDS_MIN_WINS}\r\n    per_fold = {}\r\n    for f in sorted(set(folds.tolist())):\r\n        wf = np.zeros((1, G))\r\n        wf[0, np.unique(gidx[folds == f])] = 1\r\n        per_fold[str(f)] = float(seed_mean(\"cand\", cs, \"det\", wf)[0][\"macro_f1\"][0] - seed_mean(\"ref\", rs, \"det\", wf)[0][\"macro_f1\"][0])\r\n    fwins = sum(v > 0 for v in per_fold.values())\r\n    crit[\"P-D\"] = {\"per_fold_delta\": per_fold, \"wins\": fwins, \"pass\": fwins >= FOLDS_MIN_WINS}\r\n    for name, st, cls, metric, margin in SAFEGUARDS:\r\n        dv = pdelta(st, cls, metric)\r\n        crit[name] = {\"setting\": st, \"class\": None if cls is None else CLASSES[cls], \"metric\": metric, \"delta\": dv,\r\n                      \"margin\": margin, \"ci95\": ci(name, H_LEVEL), \"pass\": dv >= -margin}\r\n    for name, cls, metric in HARM:\r\n        lo_h, hi_h = ci(name, H_LEVEL)\r\n        crit[name] = {\"class\": CLASSES[cls], \"metric\": metric, \"delta\": pdelta(\"det\", cls, metric), \"ci95\": [lo_h, hi_h],\r\n                      \"pass\": not hi_h < 0}\r\n    out[\"criteria\"] = crit\r\n    out[\"point\"] = {st: {a: {k: v[0].tolist() if np.ndim(v[0]) else float(v[0]) for k, v in point[st][a].items()} for a in point[st]}\r\n                    for st in point}\r\n    out[\"delta_b_macro_f1\"] = boot[\"macro_f1\"].tolist()\r\n    out[\"all_criteria_pass\"] = all(c[\"pass\"] for c in crit.values())\r\n    return out\r\n\r\n\r\n# ------------------------------------------------------------------------------------------------------------- verdict\r\nRUN_CLASSES = (\"ok\", \"infrastructure\", \"divergence\", \"incomplete\")\r\nN_RUNS = 25                                     # 5 folds x 5 seeds (V4)\r\n\r\n\r\ndef verdict(comparison: dict | None, *, confirmatory: bool, validity: dict, run_classes: dict, n_runs: int = N_RUNS) -> dict:\r\n    \"\"\"Section 5.4 / 9 (owner decision 2026-10-25: any divergence is FAIL; infrastructure failures are NO VERDICT).\r\n\r\n    validity: {\"V1\": bool, \"V2\": bool, \"V3\": bool, \"V5\": bool, \"V6\": bool or None (no imports)}. V1 here covers CV gates 1-3;\r\n    gates 4/5 (runs complete, one prediction per finding) and V4 are run-level and are derived from run_classes, because section 9\r\n    classifies their failure as infrastructure / incomplete. run_classes: {run: one of RUN_CLASSES} for every one of the n_runs runs;\r\n    a run absent from it counts as incomplete.\r\n\r\n    Precedence: an integrity gate (V1-3, V2, V3, V5, V6) failing -> NO VERDICT (the run evidence itself cannot be trusted);\r\n    any divergence -> FAIL, even when other runs failed for infrastructure reasons or are incomplete; infrastructure / incomplete\r\n    -> NO VERDICT; otherwise the criteria. comparison may be None when the runs never completed.\"\"\"\r\n    if not confirmatory:\r\n        raise AcceptError(\"no exploratory candidate verdicts under this protocol (section 3); only confirmatory candidates are evaluated\")\r\n    if \"V4\" in validity:\r\n        raise AcceptError(\"V4 (all runs complete) is derived from run_classes; do not pass it separately\")\r\n    missing = [g for g in (\"V1\", \"V2\", \"V3\", \"V5\") if g not in validity]\r\n    if missing:\r\n        raise AcceptError(f\"validity gates not reported: {missing}\")\r\n    unknown = sorted(r for r, c in run_classes.items() if c not in RUN_CLASSES)\r\n    if unknown:\r\n        raise AcceptError(f\"unclassified runs: {unknown}\")\r\n    if len(run_classes) > n_runs:\r\n        raise AcceptError(f\"{len(run_classes)} runs reported, expected {n_runs}\")\r\n    bad = sorted(g for g, ok in validity.items() if ok is False)\r\n    if bad:\r\n        return {\"verdict\": \"NO VERDICT\", \"reason\": f\"validity gate(s) failed: {bad}\"}\r\n    div = sorted(r for r, c in run_classes.items() if c == \"divergence\")\r\n    if div:\r\n        return {\"verdict\": \"FAIL\", \"reason\": f\"divergence in {div}\"}\r\n    inf = sorted(r for r, c in run_classes.items() if c == \"infrastructure\")\r\n    inc = sorted(r for r, c in run_classes.items() if c == \"incomplete\")\r\n    absent = n_runs - len(run_classes)\r\n    if inf or inc or absent:\r\n        return {\"verdict\": \"NO VERDICT\", \"reason\": f\"V4 failed: infrastructure failure after {MAX_ATTEMPTS} attempts {inf}, \"\r\n                                                   f\"incomplete {inc}, unreported runs {absent}\"}\r\n    if comparison is None:\r\n        raise AcceptError(\"all runs ok but no comparison supplied\")\r\n    failed = sorted(n for n, c in comparison[\"criteria\"].items() if not c[\"pass\"])\r\n    return {\"verdict\": \"FAIL\", \"reason\": f\"criteria failed: {failed}\"} if failed else {\"verdict\": \"ACCEPT\", \"reason\": \"all criteria pass\"}\r\n\r\n\r\ndef winner(candidates: list) -> dict | None:\r\n    \"\"\"Section 10.2. candidates: [{\"name\", \"verdict\", \"delta\", \"pb_lower\", \"n_params\", \"registered_at\" (ISO 8601 UTC)}].\"\"\"\r\n    ok = [c for c in candidates if c[\"verdict\"] == \"ACCEPT\"]\r\n    if not ok:\r\n        return None\r\n    return sorted(ok, key=lambda c: (-c[\"delta\"], -c[\"pb_lower\"], c[\"n_params\"], c[\"registered_at\"]))[0]\r\n\r\n\r\n# ------------------------------------------------------------------------------------------------------------- runs and budget\r\ndef collapse_share(confusion) -> float:\r\n    \"\"\"Largest predicted-class share of a run's inner-validation GT-box confusion matrix (rows true, columns predicted).\"\"\"\r\n    col = np.asarray(confusion).sum(0)\r\n    return float(col.max() / col.sum())\r\n\r\n\r\ndef classify_run(*, completed: bool, attempts: int, loss_nonfinite: bool, inner_confusion=None,\r\n                 exception_outside_compute: bool | None = None, manifest_ok: bool = True) -> str:\r\n    \"\"\"Section 9. Divergence is checked first: a NaN/inf loss (in any attempt) or a collapse is a recipe failure and is never retried.\r\n    A run that did not complete is an infrastructure failure only if its exception was raised outside the forward / backward / loss\r\n    computation (CUDA runtime error, out of memory, timeout, lost session); it is retried up to MAX_ATTEMPTS. Any other non-completion\r\n    is \"incomplete\" (gate 4 fails for a reason not covered above) and is not retried. A completed run whose genuine training\r\n    manifest is missing or does not verify (section 10.3 condition 1) is \"incomplete\": its evidence is not usable.\"\"\"\r\n    if loss_nonfinite:\r\n        return \"divergence\"\r\n    if completed:\r\n        if inner_confusion is None:\r\n            raise AcceptError(\"completed run without its inner-validation confusion matrix\")\r\n        if collapse_share(inner_confusion) >= COLLAPSE_SHARE:\r\n            return \"divergence\"\r\n        return \"ok\" if manifest_ok else \"incomplete\"\r\n    if exception_outside_compute is None:\r\n        raise AcceptError(\"run did not complete: state whether its exception was raised outside the forward/backward/loss computation\")\r\n    if not exception_outside_compute:\r\n        return \"incomplete\"\r\n    if attempts >= MAX_ATTEMPTS:\r\n        return \"infrastructure\"\r\n    raise AcceptError(f\"run not complete after {attempts} attempt(s): retry with the identical config and seed\")\r\n\r\n\r\ndef budget(registry: list) -> dict:\r\n    \"\"\"Section 9 / 10.1. registry entries: {\"name\", \"family\", \"status\"}; status one of \"registered\", \"started\" (training began;\r\n    any later outcome), \"withdrawn\", \"pre_training_validity_failure\". A slot is used by started and withdrawn candidates; a\r\n    registered, not yet started candidate reserves its slot (it cannot be withdrawn for free); a pre-training validity failure frees it.\"\"\"\r\n    used, reserved = {f: 0 for f in FAMILY_CAP}, {f: 0 for f in FAMILY_CAP}\r\n    for e in registry:\r\n        if e[\"family\"] not in FAMILY_CAP:\r\n            raise AcceptError(f\"{e['name']}: unknown family {e['family']}\")\r\n        if e[\"status\"] in (\"started\", \"withdrawn\"):\r\n            used[e[\"family\"]] += 1\r\n        elif e[\"status\"] == \"registered\":\r\n            reserved[e[\"family\"]] += 1\r\n        elif e[\"status\"] != \"pre_training_validity_failure\":\r\n            raise AcceptError(f\"{e['name']}: unknown status {e['status']}\")\r\n    held = {f: used[f] + reserved[f] for f in FAMILY_CAP}\r\n    if any(n > FAMILY_CAP[f] for f, n in held.items()) or sum(held.values()) > K_LIFETIME:\r\n        raise AcceptError(f\"budget exceeded: {held} (caps {FAMILY_CAP}, lifetime {K_LIFETIME})\")\r\n    return {\"used\": used, \"reserved\": reserved, \"total_used\": sum(used.values()), \"remaining\": K_LIFETIME - sum(held.values())}\r\n\r\n\r\n# ------------------------------------------------------------------------------------------------------------- validity gates\r\ndef check_fold_file(path) -> dict:\r\n    got = CV.sha256_file(path)\r\n    return {\"sha256\": got, \"expected\": CV.FOLD_FILE_SHA256, \"pass\": got == CV.FOLD_FILE_SHA256}\r\n\r\n\r\ndef gate_r(y, stored: dict, recomputed: dict) -> dict:\r\n    \"\"\"Section 4.3. stored / recomputed: {run: {\"gt\": (n, C), \"det\": (n, C) with NaN rows unmatched, \"idx\": (n,)}}; y indexed by idx.\r\n    R1 exact argmax (and identical matched set); R2 max |p diff| <= R2_TOL; R3 argmax metrics identical; ECE / NLL reported.\"\"\"\r\n    rows, mism, maxd, r3 = {}, 0, 0.0, True\r\n    if sorted(stored) != sorted(recomputed):\r\n        return {\"pass\": False, \"reason\": \"run sets differ\", \"stored\": sorted(stored), \"recomputed\": sorted(recomputed)}\r\n    for run in sorted(stored):\r\n        a, b = stored[run], recomputed[run]\r\n        if not np.array_equal(a[\"idx\"], b[\"idx\"]):\r\n            return {\"pass\": False, \"reason\": f\"{run}: finding index differs\"}\r\n        yy = np.asarray(y)[a[\"idx\"]]\r\n        rr = {}\r\n        for st in (\"gt\", \"det\"):\r\n            pa, pb = np.asarray(a[st], float), np.asarray(b[st], float)\r\n            ma, mb = ~np.isnan(pa).any(1), ~np.isnan(pb).any(1)\r\n            if not np.array_equal(ma, mb):\r\n                return {\"pass\": False, \"reason\": f\"{run} {st}: matched set differs\"}\r\n            la, lb = pa[ma].argmax(1), pb[ma].argmax(1)\r\n            n_mis = int((la != lb).sum())\r\n            d = float(np.abs(pa[ma] - pb[ma]).max()) if ma.any() else 0.0\r\n            ra, rb = DK.report(yy[ma], la, pa[ma]), DK.report(yy[ma], lb, pb[ma])\r\n            same = all(ra[k] == rb[k] for k in (\"accuracy\", \"macro_f1\", \"per_class\", \"confusion\"))\r\n            mism += n_mis\r\n            maxd = max(maxd, d)\r\n            r3 &= same\r\n            rr[st] = {\"argmax_mismatches\": n_mis, \"max_abs_prob_diff\": d, \"argmax_metrics_identical\": same,\r\n                      \"ece_diff\": rb[\"ece_10bin\"] - ra[\"ece_10bin\"], \"nll_diff\": rb[\"nll\"] - ra[\"nll\"]}\r\n        rows[run] = rr\r\n    return {\"R1_argmax_mismatches\": mism, \"R2_max_abs_prob_diff\": maxd, \"R2_tolerance\": R2_TOL, \"R3_identical\": bool(r3),\r\n            \"pass\": mism == 0 and maxd <= R2_TOL and bool(r3), \"runs\": rows}\r\n\r\n\r\ndef dhash(path, size=8) -> int:\r\n    \"\"\"Identical to scripts/check_leakage.py dhash (64-bit difference hash of a 9x8 grayscale thumbnail).\"\"\"\r\n    from PIL import Image\r\n    with Image.open(path) as im:\r\n        im = im.convert(\"L\").resize((size + 1, size), Image.BILINEAR)\r\n    a = np.asarray(im, dtype=np.int16)\r\n    return int(\"\".join(\"1\" if v else \"0\" for v in (a[:, 1:] > a[:, :-1]).flatten()), 2)\r\n\r\n\r\ndef thumb(path) -> np.ndarray:\r\n    \"\"\"Standardised 256x128 grayscale thumbnail, as in scripts/check_leakage.py (Pearson correlation = mean of the product).\"\"\"\r\n    from PIL import Image\r\n    with Image.open(path) as im:\r\n        a = np.asarray(im.convert(\"L\").resize((256, 128), Image.BILINEAR), dtype=np.float32)\r\n    return (a - a.mean()) / (a.std() + 1e-6)\r\n\r\n\r\ndef gate_v6(imports: list, protected: list, max_hamming=4, min_corr=0.97) -> dict:\r\n    \"\"\"Section 3.1 item 4. imports / protected: [{\"id\", \"group\", \"sha256\", \"path\", (\"role\")}]; protected = CV pool, calibration,\r\n    test and eval_locked images. A hit on group, SHA-256, or dHash <= 4 confirmed by correlation >= 0.97 refuses the import.\"\"\"\r\n    pg = {p[\"group\"]: p for p in protected}\r\n    ps = {p[\"sha256\"]: p for p in protected}\r\n    ph = [(dhash(p[\"path\"]), p) for p in protected]\r\n    thumbs = {}\r\n\r\n    def th(r):\r\n        if r[\"path\"] not in thumbs:\r\n            thumbs[r[\"path\"]] = thumb(r[\"path\"])\r\n        return thumbs[r[\"path\"]]\r\n\r\n    hits = []\r\n    for r in imports:\r\n        if r[\"group\"] in pg:\r\n            hits.append({\"import\": r[\"id\"], \"rule\": \"group\", \"protected\": pg[r[\"group\"]][\"id\"]})\r\n        if r[\"sha256\"] in ps:\r\n            hits.append({\"import\": r[\"id\"], \"rule\": \"sha256\", \"protected\": ps[r[\"sha256\"]][\"id\"]})\r\n        h = dhash(r[\"path\"])\r\n        for hp, p in ph:\r\n            if bin(h ^ hp).count(\"1\") <= max_hamming and p[\"sha256\"] != r[\"sha256\"]:\r\n                c = float((th(r) * th(p)).mean())\r\n                if c >= min_corr:\r\n                    hits.append({\"import\": r[\"id\"], \"rule\": \"near_duplicate\", \"protected\": p[\"id\"], \"correlation\": c})\r\n    return {\"imports\": len(imports), \"protected\": len(protected), \"hits\": hits, \"pass\": not hits,\r\n            \"rule\": f\"group identity; SHA-256; dHash Hamming <= {max_hamming} confirmed by thumbnail correlation >= {min_corr}\"}\r\n\r\n\r\n# ------------------------------------------------------------------------------------------------------------- V-a\r\ndef training_manifest(*, run: str, fold: int, seed: int, cfg: dict, fit_rows: list, inner_rows: list, imports: list,\r\n                      checkpoint_path, fold_file_sha256: str, best_epoch: int, fit_arrays_sha256: str,\r\n                      inner_arrays_sha256: str, trainer: str = \"dentamind_kaggle:train_run\") -> dict:\r\n    \"\"\"Per-run training manifest written by the run itself right after its checkpoint is saved (section 10.3 condition 1).\r\n    Rows are split-CSV rows (project_image_id, image_sha256, image_path, source_dataset); imports likewise. fit / inner arrays are\r\n    the exact crop arrays the run trained and selected its epoch on (their file SHA-256).\"\"\"\r\n    def imgs(rows):\r\n        seen = {}\r\n        for r in rows:\r\n            k = r[\"project_image_id\"]\r\n            seen.setdefault(k, {\"image_id\": k, \"content_sha256\": r[\"image_sha256\"], \"file_path\": r[\"image_path\"],\r\n                                \"dataset_source\": r[\"source_dataset\"], \"n_items\": 0})[\"n_items\"] += 1\r\n        return list(seen.values())\r\n    images = {\"train\": imgs(list(fit_rows) + list(imports)), \"val\": imgs(inner_rows)}\r\n    return {\"format\": \"dentamind-training-manifest-v1\", \"created\": datetime.now(timezone.utc).isoformat(), \"task\": \"findings_classification\",\r\n            \"annotation_level\": \"finding\", \"images\": images, \"counts\": {k: len(v) for k, v in images.items()},\r\n            \"seed\": seed, \"training_config\": cfg, \"cv\": {\"run\": run, \"fold\": fold, \"seed\": seed, \"fold_file_sha256\": fold_file_sha256,\r\n                                                          \"best_epoch\": int(best_epoch), \"fit_arrays_sha256\": fit_arrays_sha256,\r\n                                                          \"inner_arrays_sha256\": inner_arrays_sha256, \"trainer\": trainer,\r\n                                                          \"imports\": len(imports), \"protocol\": PROTOCOL},\r\n            \"checkpoint_file\": Path(checkpoint_path).name, \"checkpoint_bytes\": Path(checkpoint_path).stat().st_size,\r\n            \"checkpoint_sha256\": CV.sha256_file(checkpoint_path),\r\n            \"excluded_by_design\": \"held-out fold, calibration and test radiographs are never read by this run's training\"}\r\n\r\n\r\ndef write_training_manifest(checkpoint_path, manifest: dict) -> dict:\r\n    \"\"\"Same file convention as dentexmodel.training.provenance.write: <checkpoint stem>.training_manifest.json beside the checkpoint.\"\"\"\r\n    p = Path(checkpoint_path)\r\n    m = p.with_name(p.stem + \".training_manifest.json\")\r\n    m.write_text(json.dumps(manifest, indent=1, default=str), encoding=\"utf-8\")\r\n    return {\"file\": m.name, \"sha256\": CV.sha256_file(m), \"format\": manifest[\"format\"], \"counts\": manifest[\"counts\"]}\r\n\r\n\r\ndef _sums(artifact_dir: Path) -> dict:\r\n    out = {}\r\n    for line in (artifact_dir / \"SHA256SUMS.txt\").read_text().splitlines():\r\n        h, name = line.split(maxsplit=1)\r\n        out[name.lstrip(\"*\").replace(\"\\\\\", \"/\")] = h\r\n    return out\r\n\r\n\r\ndef heldout_report(artifact_dir, run: str, pool_y, *, checkpoint_path, training_manifest_path, model_id: str, version: str) -> dict:\r\n    \"\"\"Section 10.3: the eval_<version>_val.json for the V-a final artifact, from STORED held-out predictions only (no inference).\r\n    pool_y: true class index of every CV pool finding in POOL order (train + validation rows of the combined splits), which is the\r\n    index space of heldout_pred.npz \"idx\". Refuses unless the run is fold 0 / seed 42, the archive files verify, and the checkpoint hash agrees in all four places.\"\"\"\r\n    a = Path(artifact_dir)\r\n    if not run.endswith(f\"_f{FINAL_FOLD}_s{FINAL_SEED}\"):\r\n        raise AcceptError(f\"{run}: the final artifact is fold {FINAL_FOLD}, seed {FINAL_SEED} (section 10.3)\")\r\n    sums = _sums(a)\r\n    for f in (f\"runs/{run}__heldout_pred.npz\", f\"runs/{run}__cv_run.json\", \"checkpoint_manifest.json\"):\r\n        if sums.get(f) != CV.sha256_file(a / f):\r\n            raise AcceptError(f\"{f} does not match SHA256SUMS.txt\")\r\n    cvr = json.loads((a / \"runs\" / f\"{run}__cv_run.json\").read_text())\r\n    ck = json.loads((a / \"checkpoint_manifest.json\").read_text())[run][\"sha256\"]\r\n    tm = json.loads(Path(training_manifest_path).read_text(encoding=\"utf-8\"))\r\n    got = CV.sha256_file(checkpoint_path)\r\n    shas = {\"checkpoint_file\": got, \"cv_run.json\": cvr[\"checkpoint_sha256\"], \"checkpoint_manifest.json\": ck,\r\n            \"training_manifest\": tm.get(\"checkpoint_sha256\")}\r\n    if len(set(shas.values())) != 1:\r\n        raise AcceptError(f\"checkpoint SHA-256 disagrees: {shas}\")\r\n    if tm.get(\"cv\", {}).get(\"trainer\", DEFAULT_TRAINER) not in REAL_TRAINERS:\r\n        raise AcceptError(f\"{run}: trained by {tm['cv']['trainer']} (pre-flight), not by the real trainer\")\r\n    if tm.get(\"cv\", {}).get(\"run\") != run or cvr[\"fold\"] != FINAL_FOLD or cvr[\"seed\"] != FINAL_SEED:\r\n        raise AcceptError(\"training manifest or cv_run.json does not describe this run\")\r\n    d = np.load(a / \"runs\" / f\"{run}__heldout_pred.npz\", allow_pickle=False)\r\n    yy = np.asarray(pool_y)[d[\"idx\"]]\r\n    gt, det = d[\"gt\"].astype(float), d[\"det\"].astype(float)\r\n    m = ~np.isnan(det).any(1)\r\n    metrics = {\"gt\": DK.report(yy, gt.argmax(1), gt), \"det_matched\": DK.report(yy[m], det[m].argmax(1), det[m])}\r\n    return {\"generated\": datetime.now(timezone.utc).isoformat(), \"model_version\": version, \"model_id\": model_id,\r\n            \"task\": \"findings_classification\", \"split\": \"held-out CV fold 0 (stored predictions)\", \"n_images\": None,\r\n            \"n_findings\": int(len(yy)), \"n_detector_matched\": int(m.sum()), \"final_test\": False, \"metrics\": metrics,\r\n            \"checkpoint_sha256\": got, \"source\": {\"artifact_dir\": str(a), \"run\": run, \"heldout_pred_sha256\": sums[f\"runs/{run}__heldout_pred.npz\"]},\r\n            \"no_new_inference\": True, \"disclosure\": DISCLOSURE, \"protocol\": PROTOCOL}\r\n\r\n\r\n# ------------------------------------------------------------------------------------------------------------- candidate runs (section 9)\r\n# A candidate run is trained in a fresh process (as DK.train_run_isolated does) by worker(), which calls the unchanged trainer\r\n# DK.train_run, records where any exception came from, and - right after the trainer has saved and hashed best.pth, in the same\r\n# process - writes that run's training manifest. run_isolated() retries only infrastructure failures, at most MAX_ATTEMPTS in total\r\n# per run (also across resumed sessions); a computation exception, a non-finite loss or a failed manifest write is final and never\r\n# retried. run_status() turns the files of one run directory into its section-9 class.\r\nINFRA_MARKERS = (\"cuda out of memory\", \"cuda error: out of memory\", \"cudnn_status_alloc_failed\", \"cublas_status_alloc_failed\",\r\n                 \"illegal memory access\", \"unspecified launch failure\", \"cuda driver version is insufficient\",\r\n                 \"no cuda gpus are available\", \"uncorrectable ecc error\", \"cuda error: launch timed out\")\r\nNOT_INFRA_MARKERS = (\"device-side assert\",)     # a kernel assert (e.g. a label out of range) is a computation fault\r\nKILLED_RETURNCODES = (-9, 137)                  # SIGKILL (OOM killer / session teardown); any other silent exit is ambiguous\r\nATTEMPTS_FILE, FAILURE_FILE, PREDICTION_FAILURES_FILE = \"run_attempts.json\", \"run_failure.json\", \"prediction_failures.json\"\r\nCHECKPOINT_NAME = \"best.pth\"\r\nMANIFEST_NAME = \"best.training_manifest.json\"\r\nDEFAULT_TRAINER = \"dentamind_kaggle:train_run\"\r\nINTERVENTION_TRAINER = \"cv5_interventions:train_run\"    # candidate configs with intervention keys in ExpConfig.extra\r\nREAL_TRAINERS = (DEFAULT_TRAINER, INTERVENTION_TRAINER)\r\nPREFLIGHT_TRAINER_ENV = \"DENTAMIND_PREFLIGHT_TRAINER\"   # pre-flight only: fake trainer \"module:function\"; runs are never eligible\r\n\r\n\r\ndef trainer_for(cfg) -> str:\r\n    \"\"\"The real trainer for a config: DK.train_run for the BASE recipe (no extra), the intervention trainer otherwise.\"\"\"\r\n    return INTERVENTION_TRAINER if getattr(cfg, \"extra\", None) else DEFAULT_TRAINER\r\n\r\n\r\ndef exception_origin(exc: BaseException) -> str:\r\n    \"\"\"Section 9. \"infrastructure\" only for clearly recognised failures of the machine rather than the recipe: out-of-memory (device\r\n    or host; torch.cuda.OutOfMemoryError), CUDA illegal memory access / unspecified launch failure, missing or broken GPU driver, ECC\r\n    error, kernel launch timeout. Anything else - including other \"CUDA error: ...\" messages and cuDNN / cuBLAS status errors that\r\n    can be caused by the computation - is \"computation\": ambiguous failures are never retried automatically. Timeouts and\r\n    processes killed by SIGKILL are classified as infrastructure by run_isolated().\"\"\"\r\n    msg = f\"{type(exc).__name__}: {exc}\".lower()\r\n    if any(m in msg for m in NOT_INFRA_MARKERS):\r\n        return \"computation\"\r\n    if isinstance(exc, MemoryError) or type(exc).__name__ == \"OutOfMemoryError\":\r\n        return \"infrastructure\"\r\n    if isinstance(exc, RuntimeError) and any(m in msg for m in INFRA_MARKERS):\r\n        return \"infrastructure\"\r\n    return \"computation\"\r\n\r\n\r\ndef nonfinite_loss(history) -> bool:\r\n    \"\"\"Section 9 (i): training loss NaN or +-inf at any logged step (the per-epoch train_loss that DK.train_run logs).\"\"\"\r\n    return any(not np.isfinite(float(h[\"train_loss\"])) for h in history or [])\r\n\r\n\r\ndef _read(p: Path, default=None):\r\n    return json.loads(p.read_text(encoding=\"utf-8\")) if p.is_file() else default\r\n\r\n\r\ndef _write(p: Path, obj) -> None:\r\n    tmp = p.with_suffix(p.suffix + \".tmp\")\r\n    tmp.write_text(json.dumps(obj, indent=1, default=str), encoding=\"utf-8\")\r\n    tmp.replace(p)\r\n\r\n\r\ndef _state_history(run_dir: Path) -> list:\r\n    \"\"\"Epoch history saved by DK.train_run in state.pt (used to detect a non-finite loss before an exception).\"\"\"\r\n    st = run_dir / \"state.pt\"\r\n    if not st.is_file():\r\n        return []\r\n    try:\r\n        import torch\r\n        return list(torch.load(st, map_location=\"cpu\", weights_only=False).get(\"hist\") or [])\r\n    except Exception:                           # unreadable state: no evidence either way\r\n        return []\r\n\r\n\r\ndef write_rows(path, fit_rows: list, inner_rows: list, imports: list = ()) -> Path:\r\n    \"\"\"The rows a fold's runs train / select their epoch on, as passed to the worker for its manifest.\"\"\"\r\n    keep = (\"sample_id\", \"project_image_id\", \"image_sha256\", \"image_path\", \"source_dataset\", \"radiograph_group\", \"target_class\")\r\n    trim = lambda rows: [{k: r[k] for k in keep if k in r} for r in rows]      # noqa: E731\r\n    p = Path(path)\r\n    _write(p, {\"fit\": trim(fit_rows), \"inner\": trim(inner_rows), \"imports\": trim(imports)})\r\n    return p\r\n\r\n\r\ndef _resolve(spec: str):\r\n    import importlib\r\n    mod, fn = spec.split(\":\")\r\n    return getattr(importlib.import_module(mod), fn)\r\n\r\n\r\ndef _write_run_manifest(spec: dict, res: dict, out: Path) -> dict:\r\n    ck = out / CHECKPOINT_NAME\r\n    if not ck.is_file():\r\n        raise AcceptError(f\"{ck} missing after training\")\r\n    if CV.sha256_file(ck) != res[\"checkpoint_sha256\"]:\r\n        raise AcceptError(\"checkpoint on disk differs from the hash the trainer recorded\")\r\n    if res[\"config\"] != spec[\"cfg\"] or int(res[\"seed\"]) != int(spec[\"seed\"]):\r\n        raise AcceptError(\"trainer result does not describe this run's config / seed\")\r\n    rows = json.loads(Path(spec[\"rows_json\"]).read_text(encoding=\"utf-8\"))\r\n    m = training_manifest(run=spec[\"run\"], fold=spec[\"fold\"], seed=spec[\"seed\"], cfg=res[\"config\"], fit_rows=rows[\"fit\"],\r\n                          trainer=spec.get(\"trainer\", DEFAULT_TRAINER),\r\n                          inner_rows=rows[\"inner\"], imports=rows.get(\"imports\", []), checkpoint_path=ck,\r\n                          fold_file_sha256=spec[\"fold_file_sha256\"], best_epoch=res[\"best_epoch\"],\r\n                          fit_arrays_sha256=CV.sha256_file(spec[\"fit_npz\"]), inner_arrays_sha256=CV.sha256_file(spec[\"inner_npz\"]))\r\n    ref = write_training_manifest(ck, m)\r\n    side = ck.with_suffix(\".json\")              # registry sidecar written by the trainer; test_ledger reads training_manifest from it\r\n    if side.is_file():\r\n        meta = json.loads(side.read_text(encoding=\"utf-8\"))\r\n        meta[\"training_manifest\"] = ref\r\n        side.write_text(json.dumps(meta, indent=1), encoding=\"utf-8\")\r\n    return ref\r\n\r\n\r\ndef worker(spec: dict) -> int:\r\n    \"\"\"One training attempt in a fresh process. Writes <run>/attempt_<n>.json: status \"completed\" (trained, manifest written),\r\n    \"exception\" (with origin) or \"manifest_failed\". Exit code 0 only for \"completed\".\"\"\"\r\n    import traceback\r\n    out = Path(spec[\"out\"]) / spec[\"run\"]\r\n    out.mkdir(parents=True, exist_ok=True)\r\n    rec_path = out / f\"attempt_{spec['attempt']}.json\"\r\n    rec = {\"attempt\": spec[\"attempt\"], \"run\": spec[\"run\"], \"fold\": spec[\"fold\"], \"seed\": spec[\"seed\"],\r\n           \"started\": datetime.now(timezone.utc).isoformat()}\r\n    load = lambda f: (lambda z: {\"X\": z[\"X\"], \"y\": z[\"y\"]})(np.load(f))     # noqa: E731  (as DK._worker)\r\n    rec[\"trainer\"] = spec.get(\"trainer\", DEFAULT_TRAINER)\r\n    rec[\"preflight\"] = rec[\"trainer\"] not in REAL_TRAINERS\r\n    if (out / \"result.json\").is_file() or (out / CHECKPOINT_NAME).is_file():\r\n        rec.update(status=\"manifest_failed\", stage=\"start\", origin=None, type=\"StaleArtifacts\",\r\n                   message=\"checkpoint or result.json present before this attempt trained: refusing a late manifest\")\r\n        _write(rec_path, rec)\r\n        return 4\r\n    try:\r\n        res = _resolve(spec.get(\"trainer\", DEFAULT_TRAINER))(DK.ExpConfig(**spec[\"cfg\"]), spec[\"seed\"], load(spec[\"fit_npz\"]),\r\n                                                             load(spec[\"inner_npz\"]), spec[\"out\"], log=print)\r\n    except Exception as e:\r\n        rec.update(status=\"exception\", stage=\"training\", origin=exception_origin(e), type=type(e).__name__, message=str(e)[:2000],\r\n                   traceback=traceback.format_exc()[-4000:], loss_nonfinite=nonfinite_loss(_state_history(out)))\r\n        _write(rec_path, rec)\r\n        return 3\r\n    rec.update(loss_nonfinite=nonfinite_loss(res.get(\"history\")), best_epoch=res.get(\"best_epoch\"),\r\n               checkpoint_sha256=res.get(\"checkpoint_sha256\"))\r\n    try:\r\n        rec[\"manifest\"] = _write_run_manifest(spec, res, out)\r\n    except Exception as e:\r\n        rec.update(status=\"manifest_failed\", stage=\"manifest\", origin=None, type=type(e).__name__, message=str(e)[:2000],\r\n                   traceback=traceback.format_exc()[-4000:])\r\n        _write(rec_path, rec)\r\n        return 4\r\n    rec[\"status\"] = \"completed\"\r\n    _write(rec_path, rec)\r\n    return 0\r\n\r\n\r\ndef _final(out: Path, cls: str, reason: str, entry: dict | None = None) -> dict:\r\n    f = {\"class\": cls, \"reason\": reason, \"recorded\": datetime.now(timezone.utc).isoformat(), \"attempt\": entry}\r\n    _write(out / FAILURE_FILE, f)\r\n    return {\"status\": \"final_failure\", **f}\r\n\r\n\r\ndef run_isolated(cfg, seed: int, fold: int, fit_npz, inner_npz, rows_json, runs_dir, *, fold_file_sha256=CV.FOLD_FILE_SHA256,\r\n                 log=print, attempts=MAX_ATTEMPTS, timeout_s=3 * 3600, trainer=None, extra_path=(), python=None) -> dict:\r\n    \"\"\"Train one candidate run with section-9 retry rules. Returns {\"status\": \"completed\" | \"final_failure\" |\r\n    \"infrastructure_exhausted\", ...}. Attempts are counted in <run>/run_attempts.json and persist across resumed sessions.\"\"\"\r\n    import dataclasses\r\n    import os\r\n    import subprocess\r\n    import sys\r\n    trainer = trainer or os.environ.get(PREFLIGHT_TRAINER_ENV) or trainer_for(cfg)\r\n    run = f\"{cfg.name}_s{seed}\"\r\n    out = Path(runs_dir) / run\r\n    out.mkdir(parents=True, exist_ok=True)\r\n    hist = _read(out / ATTEMPTS_FILE, [])\r\n    final = _read(out / FAILURE_FILE)\r\n    if final:\r\n        return {\"status\": \"final_failure\", **final}\r\n    if hist and hist[-1][\"status\"] == \"completed\":\r\n        return {\"status\": \"completed\", \"run\": run, \"attempts\": hist}\r\n    paths = [str(Path(m.__file__).resolve().parent) for m in (DK, CV, sys.modules[\"dsutil\"])] + [str(p) for p in extra_path]\r\n    env = {**os.environ, \"PYTHONPATH\": os.pathsep.join(dict.fromkeys(paths + [os.environ.get(\"PYTHONPATH\", \"\")]))}\r\n    while len(hist) < attempts:\r\n        n = len(hist) + 1\r\n        # An attempt that ended after the checkpoint (or result.json) was written but before its worker recorded completion has no\r\n        # manifest. Retrying would let the trainer skip training and the worker write a manifest late: never allowed.\r\n        left = [f for f in (CHECKPOINT_NAME, \"result.json\") if (out / f).is_file()]\r\n        if left:\r\n            return _final(out, \"incomplete\", f\"{', '.join(left)} exists without a recorded completed attempt: no manifest was \"\r\n                                             \"written by the run that created it, and none is written later\", hist[-1] if hist else None)\r\n        spec = {\"run\": run, \"fold\": fold, \"seed\": seed, \"attempt\": n, \"cfg\": dataclasses.asdict(cfg), \"fit_npz\": str(fit_npz),\r\n                \"inner_npz\": str(inner_npz), \"rows_json\": str(rows_json), \"out\": str(runs_dir), \"fold_file_sha256\": fold_file_sha256,\r\n                \"trainer\": trainer}\r\n        p = subprocess.Popen([python or sys.executable, \"-u\", str(Path(__file__).resolve()), \"--worker\", json.dumps(spec)],\r\n                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)\r\n        tail = []\r\n        try:\r\n            for line in p.stdout:\r\n                line = line.rstrip()\r\n                tail = (tail + [line])[-40:]\r\n                if line.startswith(\"  [\") or \"resumed\" in line or \"Error\" in line:\r\n                    log(line)\r\n            rc = p.wait(timeout=timeout_s)\r\n        except subprocess.TimeoutExpired:\r\n            p.kill()\r\n            rc = \"timeout\"\r\n        rec = _read(out / f\"attempt_{n}.json\")\r\n        if rc == \"timeout\":\r\n            entry = {\"attempt\": n, \"status\": \"exception\", \"stage\": \"training\", \"origin\": \"infrastructure\", \"type\": \"timeout\",\r\n                     \"loss_nonfinite\": nonfinite_loss(_state_history(out))}\r\n        elif rec is None:                       # process died without a record\r\n            killed = rc in KILLED_RETURNCODES\r\n            entry = {\"attempt\": n, \"status\": \"exception\", \"stage\": \"training\", \"origin\": \"infrastructure\" if killed else \"ambiguous\",\r\n                     \"type\": \"killed\" if killed else \"exit_without_record\", \"returncode\": rc,\r\n                     \"loss_nonfinite\": nonfinite_loss(_state_history(out))}\r\n        else:\r\n            entry = {**rec, \"returncode\": rc}\r\n        entry[\"tail\"] = tail[-15:]\r\n        hist.append(entry)\r\n        _write(out / ATTEMPTS_FILE, hist)\r\n        if entry[\"status\"] == \"completed\" and rc == 0:\r\n            log(f\"[{run}] completed on attempt {n}; manifest {entry['manifest']['sha256'][:16]}\")\r\n            return {\"status\": \"completed\", \"run\": run, \"attempts\": hist}\r\n        if entry.get(\"loss_nonfinite\"):\r\n            return _final(out, \"divergence\", \"non-finite training loss\", entry)\r\n        if entry[\"status\"] == \"manifest_failed\":\r\n            return _final(out, \"incomplete\", f\"training manifest could not be written: {entry.get('message')}\", entry)\r\n        if entry.get(\"origin\") != \"infrastructure\":\r\n            return _final(out, \"incomplete\", f\"{entry.get('origin')} failure ({entry.get('type')}): not retried\", entry)\r\n        log(f\"[{run}] attempt {n} infrastructure failure ({entry.get('type')}): \" + \" | \".join(tail[-4:]))\r\n    log(f\"[{run}] infrastructure failures in all {attempts} attempts\")\r\n    return {\"status\": \"infrastructure_exhausted\", \"run\": run, \"attempts\": hist}\r\n\r\n\r\ndef record_prediction_failure(run_dir, exc: BaseException) -> dict:\r\n    \"\"\"Held-out prediction failed after training. A computation fault is final (incomplete); an infrastructure fault may be\r\n    redone on resume, at most MAX_ATTEMPTS times in total.\"\"\"\r\n    out = Path(run_dir)\r\n    origin = exception_origin(exc)\r\n    fails = _read(out / PREDICTION_FAILURES_FILE, []) + [{\"origin\": origin, \"type\": type(exc).__name__, \"message\": str(exc)[:2000]}]\r\n    _write(out / PREDICTION_FAILURES_FILE, fails)\r\n    if origin != \"infrastructure\":\r\n        return _final(out, \"incomplete\", f\"exception during held-out prediction ({type(exc).__name__}): not retried\", fails[-1])\r\n    if len(fails) >= MAX_ATTEMPTS:\r\n        return _final(out, \"infrastructure\", f\"held-out prediction failed {len(fails)} times (infrastructure)\", fails[-1])\r\n    return {\"status\": \"retry_on_resume\", \"origin\": origin, \"failures\": len(fails)}\r\n\r\n\r\ndef verify_run_manifest(run_dir, run: str, fold: int, seed: int) -> dict:\r\n    \"\"\"The run's own manifest describes this run and this checkpoint: file present and unmodified since the worker wrote it, same\r\n    checkpoint SHA-256 as the file on disk and the trainer's record, same run / fold / seed / selected epoch / config.\"\"\"\r\n    out = Path(run_dir)\r\n    problems = []\r\n    ck, mp = out / CHECKPOINT_NAME, out / MANIFEST_NAME\r\n    res, hist = _read(out / \"result.json\"), _read(out / ATTEMPTS_FILE, [])\r\n    done = [h for h in hist if h.get(\"status\") == \"completed\"]\r\n    if not ck.is_file() or not mp.is_file() or res is None or not done:\r\n        return {\"ok\": False, \"problems\": [\"checkpoint, manifest, result.json or completed-attempt record missing\"]}\r\n    m = json.loads(mp.read_text(encoding=\"utf-8\"))\r\n    sha = CV.sha256_file(ck)\r\n    ref = done[-1].get(\"manifest\") or {}\r\n    if ref.get(\"sha256\") != CV.sha256_file(mp):\r\n        problems.append(\"manifest file differs from the one the worker wrote\")\r\n    if not (sha == m.get(\"checkpoint_sha256\") == res.get(\"checkpoint_sha256\") == done[-1].get(\"checkpoint_sha256\")):\r\n        problems.append(\"checkpoint SHA-256 disagrees between file, manifest, result.json and attempt record\")\r\n    cvb = m.get(\"cv\", {})\r\n    want = {\"run\": run, \"fold\": fold, \"seed\": seed, \"best_epoch\": res.get(\"best_epoch\")}\r\n    for k, v in want.items():\r\n        if cvb.get(k) != v:\r\n            problems.append(f\"manifest cv.{k} = {cvb.get(k)!r}, expected {v!r}\")\r\n    if m.get(\"seed\") != seed or m.get(\"training_config\") != res.get(\"config\"):\r\n        problems.append(\"manifest seed / training_config differ from result.json\")\r\n    side = _read(ck.with_suffix(\".json\"))\r\n    if side is not None and (side.get(\"training_manifest\") or {}).get(\"sha256\") != ref.get(\"sha256\"):\r\n        problems.append(\"checkpoint sidecar does not reference this manifest\")\r\n    return {\"ok\": not problems, \"problems\": problems, \"checkpoint_sha256\": sha, \"manifest_sha256\": ref.get(\"sha256\")}\r\n\r\n\r\ndef run_status(runs_dir, run: str, fold: int, seed: int) -> dict:\r\n    \"\"\"Section-9 class of one run from its directory: ok / divergence / infrastructure / incomplete (resumable or final).\"\"\"\r\n    out = Path(runs_dir) / run\r\n    base = {\"run\": run, \"fold\": fold, \"seed\": seed}\r\n    if not out.is_dir():\r\n        return {**base, \"class\": \"incomplete\", \"reason\": \"not started\", \"resumable\": True, \"final_model_eligible\": False}\r\n    final = _read(out / FAILURE_FILE)\r\n    hist = _read(out / ATTEMPTS_FILE, [])\r\n    base[\"attempts\"] = len(hist)\r\n    base[\"exception_origins\"] = [h.get(\"origin\") for h in hist if h.get(\"status\") != \"completed\"]\r\n    if final:\r\n        return {**base, \"class\": final[\"class\"], \"reason\": final[\"reason\"], \"resumable\": False, \"final_model_eligible\": False}\r\n    if hist and hist[-1][\"status\"] == \"completed\":\r\n        res = _read(out / \"result.json\")\r\n        nonfinite = nonfinite_loss(res[\"history\"]) or any(h.get(\"loss_nonfinite\") for h in hist)\r\n        man = verify_run_manifest(out, run, fold, seed)\r\n        cls = classify_run(completed=True, attempts=len(hist), loss_nonfinite=nonfinite, inner_confusion=res[\"val_gt\"][\"confusion\"],\r\n                           manifest_ok=man[\"ok\"])\r\n        st = {**base, \"class\": cls, \"manifest\": man, \"collapse_share\": collapse_share(res[\"val_gt\"][\"confusion\"]),\r\n              \"loss_nonfinite\": nonfinite, \"resumable\": False}\r\n        if cls == \"ok\" and not (out / \"heldout_pred.npz\").is_file():\r\n            st.update({\"class\": \"incomplete\", \"reason\": \"held-out prediction missing\", \"resumable\": True})\r\n        st[\"preflight\"] = any(h.get(\"preflight\") for h in hist)\r\n        st[\"final_model_eligible\"] = st[\"class\"] == \"ok\" and man[\"ok\"] and not st[\"preflight\"]\r\n        return st\r\n    nonfinite = any(h.get(\"loss_nonfinite\") for h in hist)\r\n    try:\r\n        cls = classify_run(completed=False, attempts=len(hist), loss_nonfinite=nonfinite,\r\n                           exception_outside_compute=(hist[-1].get(\"origin\") == \"infrastructure\") if hist else True)\r\n        return {**base, \"class\": cls, \"resumable\": False, \"final_model_eligible\": False}\r\n    except AcceptError as e:\r\n        return {**base, \"class\": \"incomplete\", \"reason\": str(e), \"resumable\": True, \"final_model_eligible\": False}\r\n\r\n\r\ndef collect_status(runs_dir, expected: list) -> dict:\r\n    \"\"\"expected: [(fold, seed, run_name)] for all required runs (25). run_classes feeds verdict().\"\"\"\r\n    st = {r: run_status(runs_dir, r, f, s) for f, s, r in expected}\r\n    return {\"runs\": st, \"run_classes\": {r: v[\"class\"] for r, v in st.items()}, \"n_expected\": len(expected),\r\n            \"all_ok\": all(v[\"class\"] == \"ok\" for v in st.values())}\r\n\r\n\r\nif __name__ == \"__main__\":\r\n    import sys\r\n    if len(sys.argv) == 3 and sys.argv[1] == \"--worker\":\r\n        sys.exit(worker(json.loads(sys.argv[2])))\r\n", "kaggle/cv5_interventions.py": "\"\"\"Candidate interventions for CV5 candidates (reports/cv5_amendment_01_candidate_registration.md, DRAFT; not registered).\r\n\r\nEach intervention is selected only through ``ExpConfig.extra`` and is off by default, so the BASE recipe (``extra == {}``) is the\r\nunchanged DK code path:\r\n\r\n  crop_size  (F1)  GT and detector crops resized to this many pixels instead of dsutil.CROP_SIZE (224); C1 proposes 384\r\n  crop_pad   (F1)  context padding as a fraction of box width / height instead of dsutil.CROP_PAD (0.10); C2 proposes 0.25\r\n  backbone   (F2)  \"resnet50\" (BASE) or \"convnext_tiny\" (torchvision ConvNeXt-Tiny, IMAGENET1K_V1); C3 proposes convnext_tiny\r\n\r\nkaggle/dentamind_kaggle.py and scripts/dataset/dsutil.py are NOT modified (they are in the hashed Kaggle bundle and define BASE).\r\nCrops reuse dsutil.crop_region, which already takes size and pad as arguments; the DK crop builders and DK.train_run read module\r\nconstants and DK.new_model, so the candidate path calls them with those names temporarily rebound inside crop_rule() / model_rule().\r\nTraining runs in its own worker process (cv5_accept.run_isolated), so the rebinding there never touches another run.\r\n\"\"\"\r\nfrom __future__ import annotations\r\n\r\nimport contextlib\r\nimport dataclasses\r\nimport json\r\nfrom pathlib import Path\r\n\r\nimport dentamind_kaggle as DK\r\nimport dsutil\r\n\r\nTRAINER = \"cv5_interventions:train_run\"\r\nBASE_NEW_MODEL = DK.new_model           # bound once at import: model_rule() rebinds DK.new_model, BASE construction never recurses\r\nDEFAULTS = {\"crop_size\": dsutil.CROP_SIZE, \"crop_pad\": dsutil.CROP_PAD, \"backbone\": \"resnet50\"}\r\nFAMILY = {\"crop_size\": \"F1\", \"crop_pad\": \"F1\", \"backbone\": \"F2\"}\r\nBACKBONES = (\"resnet50\", \"convnext_tiny\")\r\nPROPOSED = {\"CROP384\": {\"crop_size\": 384}, \"PAD25\": {\"crop_pad\": 0.25}, \"CNXTT\": {\"backbone\": \"convnext_tiny\"}}\r\nBASE_FIELDS_COMPARED = [f.name for f in dataclasses.fields(DK.ExpConfig) if f.name not in (\"name\", \"note\", \"extra\")]\r\n\r\n\r\nclass InterventionError(ValueError):\r\n    pass\r\n\r\n\r\ndef resolve(extra: dict | None) -> dict:\r\n    \"\"\"Validated intervention settings: DEFAULTS overridden by the keys present in extra; unknown keys or values refuse.\"\"\"\r\n    extra = dict(extra or {})\r\n    unknown = sorted(set(extra) - set(DEFAULTS))\r\n    if unknown:\r\n        raise InterventionError(f\"unknown intervention key(s) {unknown}; allowed {sorted(DEFAULTS)}\")\r\n    out = {**DEFAULTS, **extra}\r\n    if not (isinstance(out[\"crop_size\"], int) and not isinstance(out[\"crop_size\"], bool) and 32 <= out[\"crop_size\"] <= 1024):\r\n        raise InterventionError(f\"crop_size must be an int in 32..1024, got {out['crop_size']!r}\")\r\n    if not (isinstance(out[\"crop_pad\"], (int, float)) and not isinstance(out[\"crop_pad\"], bool) and 0 <= out[\"crop_pad\"] <= 1):\r\n        raise InterventionError(f\"crop_pad must be a number in 0..1, got {out['crop_pad']!r}\")\r\n    if out[\"backbone\"] not in BACKBONES:\r\n        raise InterventionError(f\"backbone must be one of {BACKBONES}, got {out['backbone']!r}\")\r\n    return out\r\n\r\n\r\ndef config_diff(cfg) -> dict:\r\n    \"\"\"Differences from EXPERIMENTS[\"BASE\"]: ExpConfig fields (name / note excluded) plus intervention keys that differ from DEFAULTS.\"\"\"\r\n    base = DK.EXPERIMENTS[\"BASE\"]\r\n    d = {f: (getattr(base, f), getattr(cfg, f)) for f in BASE_FIELDS_COMPARED if getattr(cfg, f) != getattr(base, f)}\r\n    r = resolve(cfg.extra)\r\n    d.update({f\"extra.{k}\": (DEFAULTS[k], r[k]) for k in DEFAULTS if r[k] != DEFAULTS[k]})\r\n    return d\r\n\r\n\r\ndef check_single_intervention(cfg) -> dict:\r\n    \"\"\"V3 config-diff gate (protocol section 5.3): exactly one intervention key, nothing else changed. Returns {key, family, value}.\"\"\"\r\n    d = config_diff(cfg)\r\n    if len(d) != 1 or not next(iter(d)).startswith(\"extra.\"):\r\n        raise InterventionError(f\"a candidate must differ from BASE in exactly one intervention key; differences: {d}\")\r\n    key = next(iter(d))[len(\"extra.\"):]\r\n    return {\"key\": key, \"family\": FAMILY[key], \"value\": d[f\"extra.{key}\"][1]}\r\n\r\n\r\ndef proposed_config(cid: str, data: str = \"dentex\"):\r\n    \"\"\"ExpConfig of a proposed candidate (BASE recipe + its one intervention key). data: the BASE-P2 data option once registered.\"\"\"\r\n    if cid not in PROPOSED:\r\n        raise InterventionError(f\"unknown proposed candidate {cid!r}; proposed: {sorted(PROPOSED)}\")\r\n    return dataclasses.replace(DK.EXPERIMENTS[\"BASE\"], name=cid, data=data, extra=dict(PROPOSED[cid]),\r\n                               note=f\"CV5 candidate {cid} (DRAFT Amendment 01): BASE + {PROPOSED[cid]}\")\r\n\r\n\r\ndef crop_tag(extra) -> str:\r\n    r = resolve(extra)\r\n    return f\"s{r['crop_size']}_p{r['crop_pad']:.2f}\"\r\n\r\n\r\n@contextlib.contextmanager\r\ndef crop_rule(extra):\r\n    \"\"\"Rebind DK.CROP_SIZE / DK.CROP_PAD for the duration of a DK crop-builder call (restored on exit, also on error).\"\"\"\r\n    r = resolve(extra)\r\n    old = DK.CROP_SIZE, DK.CROP_PAD\r\n    DK.CROP_SIZE, DK.CROP_PAD = r[\"crop_size\"], r[\"crop_pad\"]\r\n    try:\r\n        yield r\r\n    finally:\r\n        DK.CROP_SIZE, DK.CROP_PAD = old\r\n\r\n\r\ndef build_crops(rows, data_root, cache_file, extra):\r\n    \"\"\"DK.build_crops with the candidate's crop rule. The cache file name must carry crop_tag(extra) (DK's cache checks ids only).\"\"\"\r\n    if cache_file and crop_tag(extra) not in Path(cache_file).name:\r\n        raise InterventionError(f\"cache file {cache_file} does not carry the crop tag {crop_tag(extra)}\")\r\n    with crop_rule(extra):\r\n        return DK.build_crops(rows, data_root, cache_file)\r\n\r\n\r\ndef build_detector_crops(rows, data_root, det_cache_path, image_id_of, extra):\r\n    with crop_rule(extra):\r\n        return DK.build_detector_crops(rows, data_root, det_cache_path, image_id_of)\r\n\r\n\r\ndef new_model(torch, tv, extra, pretrained=True):\r\n    \"\"\"BASE: exactly DK.new_model. convnext_tiny: torchvision ConvNeXt-Tiny with its last classifier layer replaced by a\r\n    len(CLASSES)-way linear layer (same class order as BASE).\"\"\"\r\n    r = resolve(extra)\r\n    if r[\"backbone\"] == \"resnet50\":\r\n        return BASE_NEW_MODEL(torch, tv, pretrained=pretrained)\r\n    m = tv.models.convnext_tiny(weights=tv.models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if pretrained else None)\r\n    m.classifier[2] = torch.nn.Linear(m.classifier[2].in_features, len(DK.CLASSES))\r\n    return m\r\n\r\n\r\n@contextlib.contextmanager\r\ndef model_rule(extra):\r\n    \"\"\"Rebind DK.new_model so DK.train_run builds the candidate's backbone (restored on exit).\"\"\"\r\n    old = DK.new_model\r\n    DK.new_model = lambda torch, tv, pretrained=True: new_model(torch, tv, extra, pretrained=pretrained)  # noqa: E731\r\n    try:\r\n        yield\r\n    finally:\r\n        DK.new_model = old\r\n\r\n\r\ndef train_run(cfg, seed, train, val, out_dir, log=print):\r\n    \"\"\"Trainer for candidate runs (selected by cv5_accept for configs with intervention keys). BASE config -> DK.train_run unchanged.\r\n    The sidecar DK writes says resnet50 / 224 / 0.10; it is corrected to the run's actual architecture and crop rule.\"\"\"\r\n    r = resolve(cfg.extra)\r\n    if r == DEFAULTS:\r\n        return DK.train_run(cfg, seed, train, val, out_dir, log=log)\r\n    if train[\"X\"].shape[1:3] != (r[\"crop_size\"], r[\"crop_size\"]) or val[\"X\"].shape[1:3] != (r[\"crop_size\"], r[\"crop_size\"]):\r\n        raise InterventionError(f\"crop arrays {train['X'].shape[1:3]} / {val['X'].shape[1:3]} do not match crop_size {r['crop_size']}\")\r\n    with crop_rule(cfg.extra), model_rule(cfg.extra):\r\n        res = DK.train_run(cfg, seed, train, val, out_dir, log=log)\r\n    side = Path(res[\"checkpoint\"]).with_suffix(\".json\")\r\n    meta = json.loads(side.read_text(encoding=\"utf-8\"))\r\n    meta.update({\"architecture\": r[\"backbone\"], \"image_size\": r[\"crop_size\"], \"crop_pad\": r[\"crop_pad\"],\r\n                 \"pretrained\": \"torchvision IMAGENET1K_V1 (ConvNeXt-Tiny)\" if r[\"backbone\"] == \"convnext_tiny\" else meta.get(\"pretrained\"),\r\n                 \"intervention\": {k: r[k] for k in DEFAULTS if r[k] != DEFAULTS[k]},\r\n                 \"trained_by\": \"kaggle/cv5_interventions.py train_run (CV5 candidate path)\"})\r\n    side.write_text(json.dumps(meta, indent=1), encoding=\"utf-8\")\r\n    return res\r\n", "kaggle/cv5_preflight_trainer.py": "\"\"\"PRE-FLIGHT ONLY: stand-in for dentamind_kaggle.train_run used by kaggle/dentamind_cv5_preflight.ipynb.\n\nIt trains nothing. Each call saves a *randomly initialised* (untrained) ResNet-50 state dict as best.pth - so the candidate notebook's\nreal held-out prediction code can load it - and writes a result.json / sidecar with the fields DK.train_run writes. The behaviour of\neach run comes from the JSON plan named by $DENTAMIND_PREFLIGHT_PLAN ({run: mode}); it scripts the section-9 cases:\n\n  ok                     completed run\n  infra_once             first attempt: CUDA out-of-memory (infrastructure, retried), then ok\n  killed                 every attempt killed by SIGKILL (infrastructure, exhausted after 3 attempts)\n  compute                ValueError (computation: final, not retried)\n  ambiguous_cuda         \"CUDA error: invalid argument\" (ambiguous: final, not retried)\n  inf_loss               completes with an infinite training loss (divergence)\n  collapse               completes with every inner-validation prediction in one class (divergence)\n  kill_after_checkpoint  saves best.pth and result.json, then is killed before the worker writes the manifest\n\nRun with cv5_accept.run_isolated; cv5_accept records the trainer name, and such runs are never eligible for final-model selection.\n\"\"\"\nimport dataclasses\nimport hashlib\nimport json\nimport os\nimport signal\nfrom pathlib import Path\n\n\ndef _sha(p):\n    return hashlib.sha256(Path(p).read_bytes()).hexdigest()\n\n\ndef _kill():\n    if hasattr(signal, \"SIGKILL\"):\n        os.kill(os.getpid(), signal.SIGKILL)\n    os._exit(137)                                  # Windows: same return-code convention\n\n\ndef train(cfg, seed, train, val, out_dir, log=print):\n    import torch\n    import torchvision as tv\n    import dentamind_kaggle as DK\n    run = f\"{cfg.name}_s{seed}\"\n    plan = json.loads(Path(os.environ[\"DENTAMIND_PREFLIGHT_PLAN\"]).read_text(encoding=\"utf-8\"))\n    mode = plan.get(run, \"ok\")\n    out = Path(out_dir) / run\n    out.mkdir(parents=True, exist_ok=True)\n    calls = out / \"preflight_calls.txt\"\n    n = int(calls.read_text()) + 1 if calls.is_file() else 1\n    calls.write_text(str(n))\n    print(f\"  [{run}] PREFLIGHT fake trainer, mode {mode}, call {n} (no training)\", flush=True)\n    if mode == \"infra_once\" and n == 1:\n        raise RuntimeError(\"CUDA out of memory. Tried to allocate 2.00 GiB (PREFLIGHT simulated)\")\n    if mode == \"killed\":\n        _kill()\n    if mode == \"compute\":\n        raise ValueError(\"PREFLIGHT simulated computation error: shape mismatch in loss\")\n    if mode == \"ambiguous_cuda\":\n        raise RuntimeError(\"CUDA error: invalid argument (PREFLIGHT simulated)\")\n    torch.manual_seed(1000 * seed + int(cfg.name.rsplit(\"_f\", 1)[1]))\n    model = DK.new_model(torch, tv, pretrained=False)               # random initialisation, never trained\n    ck = out / \"best.pth\"\n    torch.save(model.state_dict(), ck)\n    sha = _sha(ck)\n    hist = [{\"epoch\": e, \"train_loss\": (float(\"inf\") if mode == \"inf_loss\" and e == 2 else 1.0 / e)} for e in (1, 2, 3)]\n    y = [int(t) for t in val[\"y\"].tolist()]\n    pred = [0 if mode == \"collapse\" else (t if i % 3 else (t + 1) % len(DK.CLASSES)) for i, t in enumerate(y)]   # scripted, not a model\n    res = {\"experiment\": cfg.name, \"seed\": seed, \"config\": dataclasses.asdict(cfg), \"best_epoch\": 2,\n           \"val_gt\": DK.report(y, pred), \"history\": hist, \"checkpoint\": str(ck),\n           \"checkpoint_sha256\": sha, \"preflight\": \"random initialisation, not trained; metrics are not results\",\n           \"split_note\": \"PREFLIGHT synthetic arrays only\"}\n    ck.with_suffix(\".json\").write_text(json.dumps({\"version\": f\"preflight_{run}\", \"checkpoint_sha256_full\": sha, \"candidate\": True,\n                                                    \"promoted\": False, \"trained_by\": \"kaggle/cv5_preflight_trainer.py (NOT TRAINED)\"}))\n    (out / \"result.json\").write_text(json.dumps(res))\n    if mode == \"kill_after_checkpoint\":\n        _kill()\n    return res\n", "scripts/dataset/dsutil.py": "\"\"\"Shared helpers for the dataset-integration pipeline (scripts/dataset/*) and the Kaggle notebook.\n\nStandard library + numpy + Pillow only (no ``dentexmodel`` import), so the same code runs locally and on Kaggle.\n\nLabel policy (enforced by ``canonical_finding``): only names that are *the same DENTEX diagnosis* map to a target class.\nAnything else (other taxonomies, generic \"lesion\"/\"infection\", healthy-tooth labels, treatment labels) returns ``None`` and is\nkept in the provenance record with the reason. Unlabelled teeth are never turned into a class.\n\"\"\"\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\nCLASSES = [\"Caries\", \"Deep Caries\", \"Impacted\", \"Periapical Lesion\"]       # index order used by the deployed classifier\nCROP_PAD = 0.10                                                            # == dentexmodel.training.trainers.CROP_PAD\nCROP_SIZE = 224\n\n# Exact DENTEX English category names (training and official validation JSON: categories_3) -> target class.\n_DENTEX_NAMES = {\"caries\": \"Caries\", \"deep caries\": \"Deep Caries\", \"deepcaries\": \"Deep Caries\", \"impacted\": \"Impacted\",\n                 \"periapical lesion\": \"Periapical Lesion\", \"periapicallesion\": \"Periapical Lesion\"}\n\n\ndef canonical_finding(name: str | None, taxonomy: str) -> str | None:\n    \"\"\"Target class for a label *of a known taxonomy*, else None. ``taxonomy`` must be one this function knows; an unknown\n    taxonomy raises, so a new source cannot silently reuse the DENTEX mapping.\"\"\"\n    if taxonomy == \"dentex_en\":\n        return _DENTEX_NAMES.get(\" \".join(str(name or \"\").lower().replace(\"_\", \" \").split()))\n    if taxonomy == \"dentex_tr_labelme\":\n        # The Turkish label scheme in DENTEX test_data.zip is NOT mapped: on the one radiograph also present in the English-labelled\n        # training set (test_181 == train_332) the two label sets disagree (see reports/additional_dataset_candidates.md).\n        return None\n    raise ValueError(f\"unknown taxonomy {taxonomy!r}: add an explicit, documented mapping first\")\n\n\ndef sha256_file(path, chunk=1 << 20) -> str:\n    h = hashlib.sha256()\n    with open(path, \"rb\") as fh:\n        for b in iter(lambda: fh.read(chunk), b\"\"):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef sha256_bytes(b: bytes) -> str:\n    return hashlib.sha256(b).hexdigest()\n\n\ndef dhash(path, size=8) -> int:\n    \"\"\"64-bit difference hash of a 9x8 grayscale thumbnail (same definition as scripts/check_leakage.py).\"\"\"\n    from PIL import Image\n    with Image.open(path) as im:\n        im = im.convert(\"L\").resize((size + 1, size), Image.BILINEAR)\n    a = np.asarray(im, dtype=np.int16)\n    return int(\"\".join(\"1\" if v else \"0\" for v in (a[:, 1:] > a[:, :-1]).flatten()), 2)\n\n\ndef thumb(path):\n    \"\"\"Standardised 256x128 grayscale thumbnail for the pixel-correlation confirmation stage.\"\"\"\n    from PIL import Image\n    with Image.open(path) as im:\n        a = np.asarray(im.convert(\"L\").resize((256, 128), Image.BILINEAR), dtype=np.float32)\n    return (a - a.mean()) / (a.std() + 1e-6)\n\n\ndef near_duplicate_pairs(items, max_hamming=4, min_corr=0.97, thumb_fn=thumb):\n    \"\"\"items: list of (key, sha256, dhash, path). Two-stage rule used by the project's leakage check: candidate pairs with\n    Hamming distance <= ``max_hamming`` and different SHA-256, confirmed when the thumbnail Pearson correlation >= ``min_corr``.\n    Returns (confirmed [(key_a, key_b, hamming, corr)], n_candidates, max_candidate_corr).\"\"\"\n    if len(items) < 2:\n        return [], 0, None\n    arr = np.array([it[2] for it in items], dtype=np.uint64)\n    cand = []\n    for i in range(len(items) - 1):\n        x = arr[i] ^ arr[i + 1:]\n        c = np.array([bin(int(v)).count(\"1\") for v in x])\n        for j in np.where(c <= max_hamming)[0]:\n            a, b = items[i], items[i + 1 + j]\n            if a[1] != b[1]:\n                cand.append((a, b, int(c[j])))\n    cache, out, cors = {}, [], []\n    for a, b, d in cand:\n        for it in (a, b):\n            if it[0] not in cache:\n                cache[it[0]] = thumb_fn(it[3])\n        r = float((cache[a[0]] * cache[b[0]]).mean())\n        cors.append(r)\n        if r >= min_corr:\n            out.append((a[0], b[0], d, round(r, 4)))\n    return out, len(cand), (max(cors) if cors else None)\n\n\ndef crop_box_xyxy(bbox_xywh, width, height, pad=CROP_PAD):\n    \"\"\"Padded crop rectangle exactly as ``trainers.crop_region`` computes it (before resizing), clipped to the image.\"\"\"\n    x, y, w, h = (float(v) for v in bbox_xywh)\n    x1, y1, x2, y2 = x, y, x + w, y + h\n    return [max(0.0, x1 - pad * w), max(0.0, y1 - pad * h), min(float(width), x2 + pad * w), min(float(height), y2 + pad * h)]\n\n\ndef crop_region(img_u8, box_xyxy, size=CROP_SIZE, pad=CROP_PAD):\n    \"\"\"Copy of ``dentexmodel.training.trainers.crop_region`` (kept identical; tests/test_dataset_integration.py checks this).\"\"\"\n    from PIL import Image\n    H, W = img_u8.shape[:2]\n    x1, y1, x2, y2 = (float(v) for v in box_xyxy)\n    w, h = x2 - x1, y2 - y1\n    x1, y1, x2, y2 = max(0, x1 - pad * w), max(0, y1 - pad * h), min(W, x2 + pad * w), min(H, y2 + pad * h)\n    if x2 - x1 < 4 or y2 - y1 < 4:\n        return None\n    return np.asarray(Image.fromarray(img_u8[int(y1):int(y2), int(x1):int(x2)]).resize((size, size), Image.BILINEAR))\n\n\ndef write_csv(path, rows, fields=None):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fields = fields or (list(rows[0].keys()) if rows else [])\n    with open(path, \"w\", newline=\"\", encoding=\"utf-8\") as fh:\n        w = csv.DictWriter(fh, fieldnames=fields, extrasaction=\"ignore\")\n        w.writeheader()\n        for r in rows:\n            w.writerow({k: (json.dumps(v) if isinstance(v, (list, dict)) else v) for k, v in r.items()})\n\n\ndef read_csv(path):\n    with open(path, newline=\"\", encoding=\"utf-8\") as fh:\n        return list(csv.DictReader(fh))\n\n\ndef file_sha256_short(path, n=16):\n    return sha256_file(path)[:n] if Path(path).is_file() else None\n", "scripts/dataset/make_cv_folds.py": "#!/usr/bin/env python\r\n\"\"\"P4 (approved with review gate, 2026-10-16): DRAFT grouped 5-fold assignment for future experiments. Nothing runs from this file\r\nuntil the owner has reviewed and approved the exact assignment.\r\n\r\n    .venv\\\\Scripts\\\\python.exe scripts\\\\dataset\\\\make_cv_folds.py\r\n\r\n* Units are radiograph groups (all crops of one radiograph are in one fold): data/splits/combined_train.csv + combined_validation.csv.\r\n  The frozen test split and the calibration split are excluded.\r\n* Stratified by (source family, has Periapical Lesion, has Deep Caries); within a stratum, groups are taken in a seeded random order\r\n  and given to the fold with the fewest groups of that stratum (ties: fewest findings, then lowest fold number).\r\n* Writes data/splits/cv5_folds_DRAFT.csv and reports/cv5_fold_draft_summary.md (per-fold radiographs and findings per class / source).\r\n  Adding verified labels changes the input, so the draft must be regenerated and reviewed again.\r\n\"\"\"\r\nfrom __future__ import annotations\r\n\r\nimport argparse\r\nimport hashlib\r\nimport random\r\nimport sys\r\nfrom collections import Counter, defaultdict\r\nfrom pathlib import Path\r\n\r\nsys.path.insert(0, str(Path(__file__).resolve().parent))\r\nfrom dsutil import CLASSES, read_csv, write_csv  # noqa: E402\r\n\r\nROOT = Path(__file__).resolve().parents[2]\r\nDATA = ROOT / \"data\"\r\nK, SEED = 5, 42\r\n\r\n\r\ndef assign_folds(rows, k=K, seed=SEED, with_impacted=False):\r\n    \"\"\"rows: manifest rows -> {radiograph_group: fold}. with_impacted=True adds \"has Impacted\" to the stratum (option B of\r\n    reports/cv5_fold_alternative_comparison.md, approved by the owner on 2026-10-24); False reproduces the reviewed draft.\"\"\"\r\n    g = defaultdict(list)\r\n    for r in rows:\r\n        g[r[\"radiograph_group\"]].append(r)\r\n    def stratum(rs):\r\n        src = rs[0][\"source_dataset\"].split(\":\")[0]\r\n        cls = {r[\"target_class\"] for r in rs}\r\n        key = (src, \"Periapical Lesion\" in cls, \"Deep Caries\" in cls)\r\n        return key + (\"Impacted\" in cls,) if with_impacted else key\r\n    strata = defaultdict(list)\r\n    for gid, rs in g.items():\r\n        strata[stratum(rs)].append(gid)\r\n    fold_of, per_stratum, findings = {}, defaultdict(Counter), Counter()\r\n    rng = random.Random(seed)\r\n    for key in sorted(strata, key=str):\r\n        gids = sorted(strata[key])\r\n        rng.shuffle(gids)\r\n        for gid in gids:\r\n            f = min(range(k), key=lambda i: (per_stratum[key][i], findings[i], i))\r\n            fold_of[gid] = f\r\n            per_stratum[key][f] += 1\r\n            findings[f] += len(g[gid])\r\n    return fold_of\r\n\r\n\r\ndef main():\r\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\r\n    ap.add_argument(\"--approved-v1\", action=\"store_true\",\r\n                    help=\"write the owner-approved assignment (option B, Impacted-stratified) to data/splits/cv5_folds_v1.csv\")\r\n    a = ap.parse_args()\r\n    rows = read_csv(DATA / \"splits/combined_train.csv\") + read_csv(DATA / \"splits/combined_validation.csv\")\r\n    test_groups = {r[\"radiograph_group\"] for r in read_csv(DATA / \"splits/combined_test.csv\") + read_csv(DATA / \"splits/combined_calibration.csv\")}\r\n    assert not ({r[\"radiograph_group\"] for r in rows} & test_groups), \"test / calibration radiograph in the CV pool\"\r\n    fold_of = assign_folds(rows, with_impacted=a.approved_v1)\r\n    out = []\r\n    for gid in sorted(fold_of):\r\n        rs = [r for r in rows if r[\"radiograph_group\"] == gid]\r\n        c = Counter(r[\"target_class\"] for r in rs)\r\n        out.append({\"radiograph_group\": gid, \"fold\": fold_of[gid], \"source\": rs[0][\"source_dataset\"],\r\n                    \"original_split\": rs[0][\"split\"], \"n_findings\": len(rs), **{c_.replace(\" \", \"_\"): c[c_] for c_ in CLASSES}})\r\n    p = DATA / (\"splits/cv5_folds_v1.csv\" if a.approved_v1 else \"splits/cv5_folds_DRAFT.csv\")\r\n    write_csv(p, out)\r\n    sha = hashlib.sha256(p.read_bytes()).hexdigest()\r\n    title = (\"# APPROVED grouped 5-fold assignment v1 (P4, option B: Impacted-stratified; owner approval 2026-10-24)\" if a.approved_v1\r\n             else \"# DRAFT grouped 5-fold assignment (P4): FOR REVIEW, NOT APPROVED, NOT USED\")\r\n    lines = [title, \"\",\r\n             f\"File `data/splits/{p.name}`, SHA-256 `{sha}`, seed {SEED}. Units: radiograph groups of combined_train + \"\r\n             \"combined_validation; test and calibration excluded (checked: no overlap).\", \"\",\r\n             \"| Fold | Radiographs | Findings | Caries | Deep Caries | Impacted | Periapical Lesion | PL radiographs | DC radiographs | from old val split |\",\r\n             \"|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|\"]\r\n    for f in range(K):\r\n        fr = [o for o in out if o[\"fold\"] == f]\r\n        lines.append(f\"| {f} | {len(fr)} | {sum(o['n_findings'] for o in fr)} | \" + \" | \".join(str(sum(o[c.replace(' ', '_')] for o in fr)) for c in CLASSES)\r\n                     + f\" | {sum(1 for o in fr if o['Periapical_Lesion'])} | {sum(1 for o in fr if o['Deep_Caries'])}\"\r\n                     + f\" | {sum(1 for o in fr if o['original_split'] == 'validation')} |\")\r\n    src = Counter((o[\"fold\"], o[\"source\"]) for o in out)\r\n    lines += [\"\", \"Radiographs per fold and source: \" + \"; \".join(f\"fold {f}: \" + \", \".join(f\"{s} {n}\" for (ff, s), n in sorted(src.items()) if ff == f)\r\n                                                        for f in range(K)),\r\n              \"\", \"Review points: (1) every fold has Periapical Lesion radiographs; (2) source mix per fold; (3) the old validation \"\r\n              \"radiographs are spread over folds (they are trained on in 4 of 5 folds); (4) after clinician-verified labels are added, this \"\r\n              \"draft is regenerated and needs a new review.\"]\r\n    (ROOT / (\"reports/cv5_folds_v1_summary.md\" if a.approved_v1 else \"reports/cv5_fold_draft_summary.md\")).write_text(\"\\n\".join(lines) + \"\\n\", encoding=\"utf-8\")\r\n    print(\"\\n\".join(lines))\r\n\r\n\r\nif __name__ == \"__main__\":\r\n    main()\r\n"}
EMBEDDED_SHA256 = {"kaggle/dentamind_kaggle.py": "879faa93cb313b47e3b9f03dcc94a92e40981384d790f40c604117a9042f3e3d", "kaggle/cv5_lib.py": "72084f9b7067f71f808880fb70b58e546405f3d970f0e30be52dd93bdaea6b40", "kaggle/cv5_accept.py": "f55a44ed57b0232a9be2d49255c107f40dfeaf8e695aa9488d22cf5c18ff7798", "kaggle/cv5_interventions.py": "752b0da88d18ba67c2d807e68e01afced8baf2db6dcdacc9004492fded463c62", "kaggle/cv5_preflight_trainer.py": "d511da4bd07b597ef80a9d09fa88dd192e25e3c604f28ffc012254e7e15b7a33", "scripts/dataset/dsutil.py": "debeb7f08f7e5176347fe58a5d7ada3932e35539eb973590feb9b57b7a864e30", "scripts/dataset/make_cv_folds.py": "283fa080ed97c542b942611caa5d7c64230b7b06f15a7ea7ad652c98279ecff6"}
for rel, txt in EMBEDDED.items():
    assert hashlib.sha256(txt.encode("utf-8")).hexdigest() == EMBEDDED_SHA256[rel], f"embedded file changed: {rel}"
    (CODE / rel).parent.mkdir(parents=True, exist_ok=True)
    (CODE / rel).write_text(txt, encoding="utf-8", newline="")
    log("embedded", rel, EMBEDDED_SHA256[rel][:16])
for p in (CODE / "kaggle", CODE / "scripts/dataset"):
    sys.path.insert(0, str(p))
import dentamind_kaggle as DK
import cv5_lib as CV
import cv5_accept as CA
import cv5_interventions as CI
CANDIDATE_CELLS_SHA256 = {"definitions": "437c8fbf93b28fe795db00c8e206f0acf62e1cc5ca14b1271b2c3917def59457", "training": "3c91797696b51d475a85f0d453e7ff9abc41eb0b42017c67af95bf9b52a981ba", "evidence": "6fe19d019a757d693d4f8ea752086552983732ec86ad659515dabe5d6c708478", "aggregation": "ebff98a900fb65a562a99c304020751d2de5a623577aa9bd4ee40c3ba49ee610"}
PLAN = {"CV5_PREFLIGHT_f1_s43": "infra_once", "CV5_PREFLIGHT_f2_s44": "compute", "CV5_PREFLIGHT_f2_s43": "inf_loss", "CV5_PREFLIGHT_f3_s45": "collapse", "CV5_PREFLIGHT_f3_s42": "ambiguous_cuda", "CV5_PREFLIGHT_f4_s44": "kill_after_checkpoint", "CV5_PREFLIGHT_f4_s43": "killed"}
EXPECTED_CLASS = {"ok": "ok", "infra_once": "ok", "compute": "incomplete", "inf_loss": "divergence", "collapse": "divergence", "ambiguous_cuda": "incomplete", "kill_after_checkpoint": "incomplete", "killed": "infrastructure"}
EXPECTED_ATTEMPTS = {"ok": 1, "infra_once": 2, "compute": 1, "inf_loss": 1, "collapse": 1, "ambiguous_cuda": 1, "kill_after_checkpoint": 1, "killed": 3}
(ART / "preflight_plan.json").write_text(json.dumps(PLAN, indent=1))
os.environ["DENTAMIND_PREFLIGHT_PLAN"] = str(ART / "preflight_plan.json")
os.environ[CA.PREFLIGHT_TRAINER_ENV] = "cv5_preflight_trainer:train"
log("pre-flight trainer:", os.environ[CA.PREFLIGHT_TRAINER_ENV], "(saves untrained random weights; nothing is trained)")
# ---- candidate definition block (verbatim from make_cv5_notebook.candidate_cells) ----
CANDIDATE_ID = 'PREFLIGHT'
CANDIDATE = {'name': 'PREFLIGHT', 'loss': 'ce', 'weights': 'inverse_cap5', 'sampler': 'none', 'data': 'dentex', 'gamma': 2.0, 'beta': 0.999, 'epochs': 15, 'batch_size': 32, 'lr': 0.0003, 'wd': 0.0001, 'hflip': 0.5, 'contrast': 0.1, 'note': 'PREFLIGHT: fake trainer, no training', 'extra': {'crop_pad': 0.25}}

def RUN_NAME(k):
    return f"CV5_{CANDIDATE_ID}_f{k}"

def CAND_CFG(k):
    return dataclasses.replace(DK.ExpConfig(**CANDIDATE), name=RUN_NAME(k))

V3 = CI.check_single_intervention(DK.ExpConfig(**CANDIDATE))
log("V3 config diff vs BASE:", V3, "- trainer", CA.trainer_for(CAND_CFG(0)))
EXPECTED_RUNS = [(k, sd, f"{RUN_NAME(k)}_s{sd}") for k in FOLDS for sd in SEEDS]
assert SMOKE or len(EXPECTED_RUNS) == 25, "all 25 runs are required"
log("candidate", CANDIDATE_ID, "runs expected:", len(EXPECTED_RUNS))

In [ ]:
# ---- P2. synthetic frame instead of folds / images / crops / detector (no DENTEX file is read) ---------------------------------
rng = np.random.default_rng(0)
C = len(DK.CLASSES)
def _rows(k, part, n, src):
    return [{"sample_id": f"syn_f{k}_{part}_{i}", "project_image_id": f"syn_img_f{k}_{part}_{i // 2}",
             "image_sha256": hashlib.sha256(f"syn{k}{part}{i}".encode()).hexdigest(), "image_path": f"synthetic/f{k}/{part}/{i}.png",
             "source_dataset": src, "radiograph_group": f"syn_g_f{k}_{part}_{i // 2}", "target_class": DK.CLASSES[i % C],
             "split": "train"} for i in range(n)]
PLANS = {k: {"fold": k, "fit": _rows(k, "fit", 16, CV.PRIMARY), "inner": _rows(k, "inner", 8, CV.PRIMARY),
              "heldout": _rows(k, "heldout", 6, CV.PRIMARY), "heldout_secondary": _rows(k, "sec", 2, CV.SECONDARY)} for k in FOLDS}
POOL = [r for k in FOLDS for part in ("fit", "inner", "heldout", "heldout_secondary") for r in PLANS[k][part]]
PRIMARY_ROWS = [r for k in FOLDS for r in PLANS[k]["heldout"]]
SECONDARY_ROWS = [r for k in FOLDS for r in PLANS[k]["heldout_secondary"]]
IDX = {r["sample_id"]: i for i, r in enumerate(POOL)}
X_ALL = rng.integers(0, 256, size=(len(POOL), DK.CROP_SIZE, DK.CROP_SIZE, 3), dtype=np.uint8)
Y_ALL = np.array([DK.CLASSES.index(r["target_class"]) for r in POOL])
MATCHED = rng.random(len(POOL)) > 0.2
MI = np.flatnonzero(MATCHED)
XD = X_ALL[MI]
XD_OF = np.full(len(POOL), -1)
XD_OF[MI] = np.arange(len(MI))
SEEDS_ALL = list(SEEDS)
SEEDS = [42, 43, 44, 45]                     # seed 46 is never started
log(f"synthetic frame: {len(POOL)} synthetic findings, crops {X_ALL.shape}, training seeds {SEEDS} of {SEEDS_ALL}")

In [ ]:
# ---- P2b. keep the verbatim candidate cell sources for P3/P4 -----------------------------------------------------------
CANDIDATE_TRAINING = '# ---- 7. train the candidate: 5 folds x seeds, section-9 retry rules, per-run training manifest ------------------------------\nimport torch, torchvision as tv\ndev = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nFAILED = []\n\ndef budget_left():\n    return TIME_BUDGET_H * 3600 - (time.time() - T0)\n\ndef predict_heldout(ck, rows):\n    if not rows:\n        return np.zeros(0, int), np.zeros((0, len(DK.CLASSES))), np.zeros((0, len(DK.CLASSES)))\n    m = CI.new_model(torch, tv, CANDIDATE["extra"], pretrained=False)\n    m.load_state_dict(torch.load(ck, map_location="cpu", weights_only=True), strict=True)\n    m.to(dev)\n    ii = np.array([IDX[r["sample_id"]] for r in rows])\n    pg = DK.predict(torch, m, X_ALL[ii], dev)\n    pd = np.full_like(pg, np.nan)\n    has = MATCHED[ii]\n    if has.any():\n        pd[has] = DK.predict(torch, m, XD[XD_OF[ii[has]]], dev)\n    del m\n    if dev.type == "cuda":\n        torch.cuda.empty_cache()\n    return ii, pg, pd\n\nTAG = CI.crop_tag(CANDIDATE["extra"])\nfor k in FOLDS:\n    plan = PLANS[k]\n    take = lambda rows: {"X": X_ALL[[IDX[r["sample_id"]] for r in rows]], "y": Y_ALL[[IDX[r["sample_id"]] for r in rows]]}\n    fit_npz = DK.save_arrays(CACHE / f"f{k}_fit_{TAG}.npz", take(plan["fit"]))\n    inner_npz = DK.save_arrays(CACHE / f"f{k}_inner_{TAG}.npz", take(plan["inner"]))\n    rows_json = CA.write_rows(CACHE / f"f{k}_rows.json", plan["fit"], plan["inner"])\n    cfg = CAND_CFG(k)\n    for sd in SEEDS:\n        name = f"{cfg.name}_s{sd}"\n        out = RUNS / name\n        if (out / "heldout_pred.npz").is_file():\n            log(f"{name}: already predicted - skipped")\n            continue\n        if budget_left() < 900:\n            log("TIME BUDGET reached: no new run started. Save the output, then rerun with RESUME_FROM set.")\n            break\n        r = CA.run_isolated(cfg, sd, k, fit_npz, inner_npz, rows_json, RUNS, log=log)\n        if r["status"] != "completed":\n            FAILED.append(name)\n            log(f"{name}: {r[\'status\']} {r.get(\'class\', \'\')} - {r.get(\'reason\', \'\')}")\n            continue\n        st = CA.run_status(RUNS, name, k, sd)\n        if not (st["class"] == "incomplete" and st.get("reason") == "held-out prediction missing"):\n            FAILED.append(name)                 # divergence (collapse / non-finite loss) or manifest not verified: no prediction\n            log(f"{name}: {st[\'class\']} - manifest problems {st[\'manifest\'][\'problems\']}, collapse share {st[\'collapse_share\']:.4f}")\n            continue\n        try:\n            ii, pg, pd = predict_heldout(out / "best.pth", plan["heldout"])\n            ii2, pg2, pd2 = predict_heldout(out / "best.pth", plan["heldout_secondary"])\n        except Exception as e:\n            FAILED.append(name)\n            log(f"{name}: held-out prediction failed:", CA.record_prediction_failure(out, e))\n            continue\n        np.savez(out / "heldout_pred.npz", idx=ii, gt=pg, det=pd, idx_sec=ii2, gt_sec=pg2, det_sec=pd2)\n        res = json.loads((out / "result.json").read_text(encoding="utf-8"))\n        done = [a for a in r["attempts"] if a["status"] == "completed"][-1]\n        (out / "cv_run.json").write_text(json.dumps({"fold": k, "seed": sd, "run": name, "candidate": CANDIDATE_ID,\n            "fit_findings": len(plan["fit"]), "inner_findings": len(plan["inner"]), "heldout_findings": len(plan["heldout"]),\n            "heldout_secondary_findings": len(plan["heldout_secondary"]), "best_epoch": res["best_epoch"],\n            "inner_val_macro_f1": res["val_gt"]["macro_f1"], "checkpoint_sha256": res["checkpoint_sha256"],\n            "training_manifest": done["manifest"], "attempts": len(r["attempts"]),\n            "exception_origins": [a.get("origin") for a in r["attempts"] if a["status"] != "completed"],\n            "note": "result.json split_note is the generic BASE text; for CV the validation set is the fold\'s inner validation subset"}, indent=1))\n        log(f"{name}: best ep {res[\'best_epoch\']} inner-val macro-F1 {res[\'val_gt\'][\'macro_f1\']:.4f}  "\n            f"elapsed {(time.time() - T0) / 3600:.2f} h")\nRUN_STATUS = CA.collect_status(RUNS, EXPECTED_RUNS)\n(ART / "run_status.json").write_text(json.dumps(RUN_STATUS, indent=1, default=str))\nlog("run classes:", dict(collections.Counter(RUN_STATUS["run_classes"].values())), "- all 25 valid and complete:", RUN_STATUS["all_ok"])\nlog("failed runs:", FAILED or "none")'
CANDIDATE_EVIDENCE = '# ---- 7b. run evidence (always packaged, also when runs failed): status, attempts, failures, manifests ------------------------\nev = ART / "run_evidence"\nif ev.exists():\n    shutil.rmtree(ev)\n(ev / "runs").mkdir(parents=True)\nEVIDENCE_FILES = ("result.json", "cv_run.json", "best.json", CA.MANIFEST_NAME, CA.ATTEMPTS_FILE, CA.FAILURE_FILE, CA.PREDICTION_FAILURES_FILE)\nfor _, _, name in EXPECTED_RUNS:\n    d = RUNS / name\n    for f in EVIDENCE_FILES + tuple(p.name for p in sorted(d.glob("attempt_*.json"))) if d.is_dir() else ():\n        if (d / f).is_file():\n            shutil.copy2(d / f, ev / "runs" / f"{name}__{f}")\nfor f in ("run_status.json", "training_log.txt"):\n    if (ART / f).is_file():\n        shutil.copy2(ART / f, ev / f)\nsums = [f"{CV.sha256_file(p)}  {p.relative_to(ev).as_posix()}" for p in sorted(ev.rglob("*")) if p.is_file()]\n(ev / "SHA256SUMS.txt").write_text("\\n".join(sums) + "\\n")\nEVZ = WORK / f"dentamind_cv5_{CANDIDATE_ID}_run_evidence{\'_SMOKE\' if SMOKE else \'\'}.zip"\nwith zipfile.ZipFile(EVZ, "w", zipfile.ZIP_DEFLATED) as z:\n    for p in sorted(ev.rglob("*")):\n        if p.is_file():\n            z.write(p, p.relative_to(ev).as_posix())\nlog(f"RUN EVIDENCE {EVZ}  sha256 {CV.sha256_file(EVZ)}")'
CANDIDATE_AGGREGATION = '# ---- 8. out-of-fold evaluation (gates 4 and 5), aggregation, bootstrap, Caries vs Deep Caries ----------------------------------\nP_IDX = np.array([IDX[r["sample_id"]] for r in PRIMARY_ROWS])\nS_IDX = np.array([IDX[r["sample_id"]] for r in SECONDARY_ROWS])\npos = {g: i for i, g in enumerate(P_IDX)}\nspos = {g: i for i, g in enumerate(S_IDX)}\nn, ns, C = len(P_IDX), len(S_IDX), len(DK.CLASSES)\nOOF, OOF_S, FOLD_OF_FINDING = {}, {}, np.full(n, -1)\nmissing_runs = []\nfor sd in SEEDS:\n    o = {"gt": np.full((n, C), np.nan), "det": np.full((n, C), np.nan), "filled": np.zeros(n, int)}\n    os_ = {"gt": np.full((ns, C), np.nan), "det": np.full((ns, C), np.nan)}\n    for k in FOLDS:\n        f = RUNS / f"{RUN_NAME(k)}_s{sd}" / "heldout_pred.npz"\n        if not f.is_file():\n            missing_runs.append(f.parent.name)\n            continue\n        z = np.load(f)\n        rows_i = np.array([pos[i] for i in z["idx"]])\n        o["gt"][rows_i], o["det"][rows_i] = z["gt"], z["det"]\n        o["filled"][rows_i] += 1\n        FOLD_OF_FINDING[rows_i] = k\n        if len(z["idx_sec"]):\n            si = np.array([spos[i] for i in z["idx_sec"]])\n            os_["gt"][si], os_["det"][si] = z["gt_sec"], z["det_sec"]\n    OOF[sd], OOF_S[sd] = o, os_\nassert RUN_STATUS["all_ok"] and not FAILED and not missing_runs, (f"gate 4: not all {len(EXPECTED_RUNS)} runs are valid and "\n    f"complete (run_status.json): {FAILED + missing_runs} - no candidate number is reported")\nCV.check_oof(OOF, n, SEEDS)\nlog("gates 4 and 5 passed: every primary finding has exactly one out-of-fold prediction per seed")\n\ny = Y_ALL[P_IDX]\nm = MATCHED[P_IDX]\ngroups = np.array([r["radiograph_group"] for r in PRIMARY_ROWS])\nPOOLED = CV.pooled_reports(y, OOF, m, SEEDS)\nsettings = ["gt"] + (["det_matched"] if DET_OK else [])\nSUMMARY = {s: {"macro_f1": CV.mean_sd([POOLED[sd][s]["macro_f1"] for sd in SEEDS]),\n               "accuracy": CV.mean_sd([POOLED[sd][s]["accuracy"] for sd in SEEDS]),\n               "ece_10bin": CV.mean_sd([POOLED[sd][s]["ece_10bin"] for sd in SEEDS]),\n               "per_class": {c: {mm: CV.mean_sd([POOLED[sd][s]["per_class"][c][mm] for sd in SEEDS]) for mm in ("precision", "recall", "f1")}\n                             for c in DK.CLASSES},\n               "confusion_sum_over_seeds": np.sum([POOLED[sd][s]["confusion"] for sd in SEEDS], axis=0).tolist()} for s in settings}\nBOOT = {"gt": CV.cluster_bootstrap(y, [OOF[sd]["gt"].argmax(1) for sd in SEEDS], groups)}\nif DET_OK:\n    BOOT["det_matched"] = CV.cluster_bootstrap(y, [np.nan_to_num(OOF[sd]["det"], nan=0).argmax(1) for sd in SEEDS], groups, valid=m)\nPER_FOLD = {}\nfor k in FOLDS:\n    sub = np.flatnonzero(FOLD_OF_FINDING == k)\n    rep = CV.pooled_reports(y[sub], {sd: {kk: OOF[sd][kk][sub] for kk in ("gt", "det")} for sd in SEEDS}, m[sub], SEEDS)\n    PER_FOLD[k] = {"radiographs": int(len(set(groups[sub]))), "findings": int(len(sub)), "detector_matched": int(m[sub].sum()),\n                   **{s: {"macro_f1": CV.mean_sd([rep[sd][s]["macro_f1"] for sd in SEEDS]),\n                          "per_class": {c: {mm: CV.mean_sd([rep[sd][s]["per_class"][c][mm] for sd in SEEDS]) for mm in ("precision", "recall", "f1")}\n                                        for c in DK.CLASSES},\n                          "confusion_sum_over_seeds": np.sum([rep[sd][s]["confusion"] for sd in SEEDS], axis=0).tolist()} for s in settings}}\nFOLD_LEVEL = {s: CV.mean_sd([PER_FOLD[k][s]["macro_f1"]["mean"] for k in FOLDS]) for s in settings}\nold_val = np.array([r["split"] == "validation" for r in PRIMARY_ROWS])\nUNSEEN = CV.pooled_reports(y[old_val], {sd: {kk: OOF[sd][kk][old_val] for kk in ("gt", "det")} for sd in SEEDS}, m[old_val], SEEDS)\nDET_UNSEEN = {"population": "65 old-validation radiographs (not in the detector\'s training assignment)", "findings": int(old_val.sum()),\n              "macro_f1_det_matched": CV.mean_sd([UNSEEN[sd]["det_matched"]["macro_f1"] for sd in SEEDS]) if DET_OK else None,\n              "macro_f1_gt": CV.mean_sd([UNSEEN[sd]["gt"]["macro_f1"] for sd in SEEDS])}\nys = Y_ALL[S_IDX]\nSECONDARY = {"population": "DENTEX official-validation findings (never trained on)", "findings": int(ns),\n             "gt_macro_f1": CV.mean_sd([DK.report(ys, OOF_S[sd]["gt"].argmax(1))["macro_f1"] for sd in SEEDS])}\nCARIES = CV.caries_confusion(PRIMARY_ROWS, OOF, m, SEEDS)\n\nfor s in settings:\n    log(f"pooled out-of-fold macro-F1 ({s}): {SUMMARY[s][\'macro_f1\'][\'mean\']:.4f} +- {SUMMARY[s][\'macro_f1\'][\'sd\']:.4f} over seeds; "\n        f"radiograph bootstrap 95% CI {BOOT[s][\'ci95\'][0]:.4f}-{BOOT[s][\'ci95\'][1]:.4f}; fold-level mean {FOLD_LEVEL[s][\'mean\']:.4f} "\n        f"+- {FOLD_LEVEL[s][\'sd\']:.4f}")\nlog("Caries/Deep Caries findings confused by all seeds (GT boxes):", CARIES["confused_by_all_seeds_gt"], "of", CARIES["n"])\n\nRESULTS = {"protocol": "reports/cv5_acceptance_protocol.md", "candidate": CANDIDATE_ID, "smoke": SMOKE, "seeds": SEEDS, "folds": FOLDS, "environment": env,\n           "gates": {"1_folds": True, "2_plans": True, "3_detector": {k: v for k, v in GATE3.items()}, "4_runs_complete": True,\n                     "5_one_prediction_per_finding": True},\n           "population": {"primary_findings": int(n), "primary_radiographs": int(len(set(groups))), "detector_matched": int(m.sum())},\n           "pooled_per_seed": {str(sd): POOLED[sd] for sd in SEEDS}, "summary": SUMMARY, "bootstrap": BOOT,\n           "per_fold": {str(k): v for k, v in PER_FOLD.items()}, "fold_level_macro_f1": FOLD_LEVEL,\n           "detector_unseen_subset": DET_UNSEEN, "secondary_official_val": SECONDARY,\n           "caries_confusion": {k: v for k, v in CARIES.items() if k != "records"}}\n(ART / "cv5_results.json").write_text(json.dumps(RESULTS, indent=1, default=float))\nimport csv\nwith open(ART / "caries_confusion_findings.csv", "w", newline="", encoding="utf-8") as fh:\n    w = csv.DictWriter(fh, fieldnames=list(CARIES["records"][0]))\n    w.writeheader()\n    w.writerows(CARIES["records"])\nwith open(ART / "oof_predictions.csv", "w", newline="", encoding="utf-8") as fh:\n    w = csv.writer(fh)\n    w.writerow(["sample_id", "radiograph_group", "fold", "true_class", "detector_matched"] +\n               [f"s{sd}_{s}_{c}" for sd in SEEDS for s in ("gt", "det") for c in DK.CLASSES])\n    for i, r in enumerate(PRIMARY_ROWS):\n        w.writerow([r["sample_id"], r["radiograph_group"], int(FOLD_OF_FINDING[i]), r["target_class"], bool(m[i])] +\n                   [("" if np.isnan(v) else round(float(v), 6)) for sd in SEEDS for s in ("gt", "det") for v in OOF[sd][s][i]])'
SEEDS_TRAINED = list(SEEDS)

In [ ]:
# ---- 7. train the candidate: 5 folds x seeds, section-9 retry rules, per-run training manifest ------------------------------
import torch, torchvision as tv
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FAILED = []

def budget_left():
    return TIME_BUDGET_H * 3600 - (time.time() - T0)

def predict_heldout(ck, rows):
    if not rows:
        return np.zeros(0, int), np.zeros((0, len(DK.CLASSES))), np.zeros((0, len(DK.CLASSES)))
    m = CI.new_model(torch, tv, CANDIDATE["extra"], pretrained=False)
    m.load_state_dict(torch.load(ck, map_location="cpu", weights_only=True), strict=True)
    m.to(dev)
    ii = np.array([IDX[r["sample_id"]] for r in rows])
    pg = DK.predict(torch, m, X_ALL[ii], dev)
    pd = np.full_like(pg, np.nan)
    has = MATCHED[ii]
    if has.any():
        pd[has] = DK.predict(torch, m, XD[XD_OF[ii[has]]], dev)
    del m
    if dev.type == "cuda":
        torch.cuda.empty_cache()
    return ii, pg, pd

TAG = CI.crop_tag(CANDIDATE["extra"])
for k in FOLDS:
    plan = PLANS[k]
    take = lambda rows: {"X": X_ALL[[IDX[r["sample_id"]] for r in rows]], "y": Y_ALL[[IDX[r["sample_id"]] for r in rows]]}
    fit_npz = DK.save_arrays(CACHE / f"f{k}_fit_{TAG}.npz", take(plan["fit"]))
    inner_npz = DK.save_arrays(CACHE / f"f{k}_inner_{TAG}.npz", take(plan["inner"]))
    rows_json = CA.write_rows(CACHE / f"f{k}_rows.json", plan["fit"], plan["inner"])
    cfg = CAND_CFG(k)
    for sd in SEEDS:
        name = f"{cfg.name}_s{sd}"
        out = RUNS / name
        if (out / "heldout_pred.npz").is_file():
            log(f"{name}: already predicted - skipped")
            continue
        if budget_left() < 900:
            log("TIME BUDGET reached: no new run started. Save the output, then rerun with RESUME_FROM set.")
            break
        r = CA.run_isolated(cfg, sd, k, fit_npz, inner_npz, rows_json, RUNS, log=log)
        if r["status"] != "completed":
            FAILED.append(name)
            log(f"{name}: {r['status']} {r.get('class', '')} - {r.get('reason', '')}")
            continue
        st = CA.run_status(RUNS, name, k, sd)
        if not (st["class"] == "incomplete" and st.get("reason") == "held-out prediction missing"):
            FAILED.append(name)                 # divergence (collapse / non-finite loss) or manifest not verified: no prediction
            log(f"{name}: {st['class']} - manifest problems {st['manifest']['problems']}, collapse share {st['collapse_share']:.4f}")
            continue
        try:
            ii, pg, pd = predict_heldout(out / "best.pth", plan["heldout"])
            ii2, pg2, pd2 = predict_heldout(out / "best.pth", plan["heldout_secondary"])
        except Exception as e:
            FAILED.append(name)
            log(f"{name}: held-out prediction failed:", CA.record_prediction_failure(out, e))
            continue
        np.savez(out / "heldout_pred.npz", idx=ii, gt=pg, det=pd, idx_sec=ii2, gt_sec=pg2, det_sec=pd2)
        res = json.loads((out / "result.json").read_text(encoding="utf-8"))
        done = [a for a in r["attempts"] if a["status"] == "completed"][-1]
        (out / "cv_run.json").write_text(json.dumps({"fold": k, "seed": sd, "run": name, "candidate": CANDIDATE_ID,
            "fit_findings": len(plan["fit"]), "inner_findings": len(plan["inner"]), "heldout_findings": len(plan["heldout"]),
            "heldout_secondary_findings": len(plan["heldout_secondary"]), "best_epoch": res["best_epoch"],
            "inner_val_macro_f1": res["val_gt"]["macro_f1"], "checkpoint_sha256": res["checkpoint_sha256"],
            "training_manifest": done["manifest"], "attempts": len(r["attempts"]),
            "exception_origins": [a.get("origin") for a in r["attempts"] if a["status"] != "completed"],
            "note": "result.json split_note is the generic BASE text; for CV the validation set is the fold's inner validation subset"}, indent=1))
        log(f"{name}: best ep {res['best_epoch']} inner-val macro-F1 {res['val_gt']['macro_f1']:.4f}  "
            f"elapsed {(time.time() - T0) / 3600:.2f} h")
RUN_STATUS = CA.collect_status(RUNS, EXPECTED_RUNS)
(ART / "run_status.json").write_text(json.dumps(RUN_STATUS, indent=1, default=str))
log("run classes:", dict(collections.Counter(RUN_STATUS["run_classes"].values())), "- all 25 valid and complete:", RUN_STATUS["all_ok"])
log("failed runs:", FAILED or "none")

In [ ]:
# ---- 7b. run evidence (always packaged, also when runs failed): status, attempts, failures, manifests ------------------------
ev = ART / "run_evidence"
if ev.exists():
    shutil.rmtree(ev)
(ev / "runs").mkdir(parents=True)
EVIDENCE_FILES = ("result.json", "cv_run.json", "best.json", CA.MANIFEST_NAME, CA.ATTEMPTS_FILE, CA.FAILURE_FILE, CA.PREDICTION_FAILURES_FILE)
for _, _, name in EXPECTED_RUNS:
    d = RUNS / name
    for f in EVIDENCE_FILES + tuple(p.name for p in sorted(d.glob("attempt_*.json"))) if d.is_dir() else ():
        if (d / f).is_file():
            shutil.copy2(d / f, ev / "runs" / f"{name}__{f}")
for f in ("run_status.json", "training_log.txt"):
    if (ART / f).is_file():
        shutil.copy2(ART / f, ev / f)
sums = [f"{CV.sha256_file(p)}  {p.relative_to(ev).as_posix()}" for p in sorted(ev.rglob("*")) if p.is_file()]
(ev / "SHA256SUMS.txt").write_text("\n".join(sums) + "\n")
EVZ = WORK / f"dentamind_cv5_{CANDIDATE_ID}_run_evidence{'_SMOKE' if SMOKE else ''}.zip"
with zipfile.ZipFile(EVZ, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(ev.rglob("*")):
        if p.is_file():
            z.write(p, p.relative_to(ev).as_posix())
log(f"RUN EVIDENCE {EVZ}  sha256 {CV.sha256_file(EVZ)}")

In [ ]:
# ---- P3. aggregation must be refused: run the candidate aggregation cell verbatim --------------------------------------------
SEEDS = SEEDS_ALL
AGG_REFUSED, AGG_MESSAGE = False, None
try:
    exec(compile(CANDIDATE_AGGREGATION, "<candidate aggregation cell>", "exec"), globals())
except AssertionError as e:
    AGG_REFUSED, AGG_MESSAGE = True, str(e)
log("aggregation refused:", AGG_REFUSED, "-", AGG_MESSAGE)

In [ ]:
# ---- P4. pre-flight checks, report, output archive ---------------------------------------------------------------------------
import zipfile as _zf
checks = {}
def check(name, ok, detail=None):
    checks[name] = {"pass": bool(ok), "detail": detail}
    log(("PASS " if ok else "FAIL ") + name, "" if detail is None else detail)
st = json.loads((ART / "run_status.json").read_text())
check("run_status.json lists all 25 required runs", len(st["runs"]) == 25 and st["n_expected"] == 25, len(st["runs"]))
check("not all runs ok", st["all_ok"] is False)
for k, sd, name in EXPECTED_RUNS:
    s = st["runs"][name]
    if sd not in SEEDS_TRAINED:
        check(f"{name} never started -> incomplete", s["class"] == "incomplete" and s.get("reason") == "not started", s["class"])
        continue
    mode = PLAN.get(name, "ok")
    check(f"{name} [{mode}] class", s["class"] == EXPECTED_CLASS[mode], s["class"])
    check(f"{name} [{mode}] attempts", s.get("attempts") == EXPECTED_ATTEMPTS[mode], s.get("attempts"))
    check(f"{name} never eligible (pre-flight)", s.get("final_model_eligible") is False)
    d = RUNS / name
    mf = d / CA.MANIFEST_NAME
    if mode in ("ok", "infra_once", "inf_loss", "collapse"):
        m = json.loads(mf.read_text())
        ck = CV.sha256_file(d / "best.pth")
        v = CA.verify_run_manifest(d, name, k, sd)
        check(f"{name} manifest verifies", v["ok"], v["problems"])
        check(f"{name} manifest <-> checkpoint <-> run", m["checkpoint_sha256"] == ck and (m["cv"]["run"], m["cv"]["fold"], m["cv"]["seed"]) == (name, k, sd)
              and m["cv"]["trainer"] == "cv5_preflight_trainer:train")
        if mode in ("ok", "infra_once"):
            cvr = json.loads((d / "cv_run.json").read_text())
            check(f"{name} cv_run.json references manifest + checkpoint", cvr["checkpoint_sha256"] == ck and
                  cvr["training_manifest"]["sha256"] == CV.sha256_file(mf), cvr["exception_origins"])
            check(f"{name} held-out predictions written", (d / "heldout_pred.npz").is_file())
        else:
            check(f"{name} diverged run not predicted", not (d / "heldout_pred.npz").exists())
    else:
        check(f"{name} has no manifest", not mf.exists())
        check(f"{name} has no held-out predictions", not (d / "heldout_pred.npz").exists())
    if mode == "kill_after_checkpoint":
        check(f"{name}: checkpoint exists but run final-incomplete, no late manifest",
              (d / "best.pth").is_file() and (d / CA.FAILURE_FILE).is_file() and not mf.exists() and
              (d / "preflight_calls.txt").read_text() == "1", json.loads((d / CA.FAILURE_FILE).read_text())["reason"])
    if mode in ("compute", "ambiguous_cuda"):
        check(f"{name}: not retried", (d / "preflight_calls.txt").read_text() == "1")
    if mode == "infra_once":
        a = json.loads((d / CA.ATTEMPTS_FILE).read_text())
        check(f"{name}: origins recorded", [x.get("origin") for x in a] == ["infrastructure", None], [x.get("type") for x in a])
    if mode == "killed":
        a = json.loads((d / CA.ATTEMPTS_FILE).read_text())
        check(f"{name}: SIGKILL recorded as infrastructure", all(x["type"] == "killed" and x["returncode"] in CA.KILLED_RETURNCODES for x in a),
              [x["returncode"] for x in a])
check("aggregation refused at gate 4", AGG_REFUSED and "gate 4" in (AGG_MESSAGE or ""), AGG_MESSAGE)
check("no candidate result written", not (ART / "cv5_results.json").exists() and not (ART / "oof_predictions.csv").exists())
check("candidate cells used verbatim", all(hashlib.sha256(src.encode()).hexdigest() == CANDIDATE_CELLS_SHA256[n] for n, src in
      (("training", CANDIDATE_TRAINING), ("evidence", CANDIDATE_EVIDENCE), ("aggregation", CANDIDATE_AGGREGATION))))
with _zf.ZipFile(EVZ) as z:
    names = set(z.namelist())
    sums = dict(reversed(l.split("  ", 1)) for l in z.read("SHA256SUMS.txt").decode().splitlines())
    bad = [n for n, h in sums.items() if hashlib.sha256(z.read(n)).hexdigest() != h]
check("run-evidence archive: SHA256SUMS verify", not bad and len(sums) > 0, f"{len(sums)} files")
check("run-evidence archive has run_status.json and every started run's attempts", "run_status.json" in names and all(
      f"runs/{n}__{CA.ATTEMPTS_FILE}" in names for k, sd, n in EXPECTED_RUNS if sd in SEEDS_TRAINED))
ok_v = {"V1": True, "V2": True, "V3": True, "V5": True, "V6": None}
v1 = CA.verdict(None, confirmatory=True, validity=ok_v, run_classes=st["run_classes"])
check("verdict with divergences present: FAIL", v1["verdict"] == "FAIL", v1["reason"])
no_div = {r: ("ok" if c == "divergence" else c) for r, c in st["run_classes"].items()}
v2 = CA.verdict(None, confirmatory=True, validity=ok_v, run_classes=no_div)
check("verdict without divergence, runs missing: NO VERDICT", v2["verdict"] == "NO VERDICT", v2["reason"])
check("no DENTEX data present", not any(DATA_ROOT.rglob("*")) and not glob.glob("/kaggle/input/*"))
PASSED = all(c["pass"] for c in checks.values())
report = {"preflight": "CV5 candidate path, no training", "result": "PASS" if PASSED else "FAIL", "n_checks": len(checks),
          "n_failed": sum(not c["pass"] for c in checks.values()), "checks": checks, "environment": env,
          "embedded_sha256": EMBEDDED_SHA256, "candidate_cells_sha256": CANDIDATE_CELLS_SHA256, "plan": PLAN,
          "run_evidence_zip_sha256": CV.sha256_file(EVZ)}
(ART / "preflight_report.json").write_text(json.dumps(report, indent=1, default=str))
OUTZ = WORK / "dentamind_cv5_preflight_output.zip"
with _zf.ZipFile(OUTZ, "w", _zf.ZIP_DEFLATED) as z:
    for p in (ART / "preflight_report.json", ART / "run_status.json", ART / "training_log.txt", EVZ):
        z.write(p, p.name)
log(f"PREFLIGHT RESULT: {report['result']} ({report['n_checks']} checks, {report['n_failed']} failed)")
log(f"OUTPUT {OUTZ}  sha256 {CV.sha256_file(OUTZ)}")
removed = 0
for p in list(RUNS.glob("*/best.pth")) + list(RUNS.glob("*/state.pt")):  # untrained random weights: not kept as notebook output
    p.unlink()
    removed += 1
for p in CACHE.glob("f*_*.npz"):
    p.unlink()
log(f"removed {removed} untrained pre-flight weight files and the synthetic arrays")